# Analysis

**Hypothesis**: Within specific cardiac cell populations, local cell density and spatial neighborhood composition are systematically associated with shifts in transcriptional complexity and purity, revealing microenvironment-linked maturation or stress states that were not captured by previous gradient- and signature-based analyses.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_anon.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Within specific cardiac cell populations, local cell density and spatial neighborhood composition are systematically associated with shifts in transcriptional complexity and purity, revealing microenvironment-linked maturation or stress states that were not captured by previous gradient- and signature-based analyses.

## Steps:
- Compute per-cell local spatial density from 2D spatial coordinates, derive per-cell neighborhood composition (fraction of each Population label among k-nearest neighbors), and summarize these metrics across annotated populations and samples to identify candidate populations with strong microenvironmental structure.
- Within the most spatially structured populations (e.g., those with highest coefficient of variation in local density), quantify the association between local density and transcriptional complexity / UMI count / purity using correlation and simple regression models that account for sample effects, and report effect sizes and p-values.
- For key populations of interest, test whether transcriptional complexity and purity differ between cells in dense versus sparse microenvironments (e.g., lowest vs highest density quantiles) using non-parametric tests per population, with multiple-testing-aware summaries.
- Characterize how immediate neighborhood cell-type composition (fraction of each Population label among k-nearest spatial neighbors) relates to complexity and purity within each focal population, using per-population linear models and approximate partial correlations that control for local density.
- Identify genes whose expression within each focal population is significantly associated with local density or specific neighbor compositions using per-gene Spearman correlations and FDR correction, summarizing the top positively and negatively associated genes per population.
- Integrate results by printing a ranked table of populations showing density–complexity and density–purity association strengths, key neighborhood-composition effects, and counts of microenvironment-associated genes, to highlight which populations are most microenvironment-sensitive.


## This code computes per-cell local spatial density and k-nearest-neighbor-based neighborhood composition from 2D spatial coordinates, summarizes density variability across populations and samples, and stores all metrics and parameters in adata for reproducible downstream analyses.

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# Parameters and basic checks
required_obsm = 'spatial'
if required_obsm not in adata.obsm:
    raise KeyError(f"Required spatial coordinates not found in adata.obsm['{required_obsm}']")

coords = adata.obsm[required_obsm]
if coords.ndim != 2 or coords.shape[1] != 2:
    raise ValueError(f"Expected 2D spatial coordinates with shape (n_cells, 2), got {coords.shape}")

n_cells = coords.shape[0]
k = 15  # number of neighbors for local density / neighborhood estimates (balance locality vs stability)
if k > n_cells:
    raise ValueError(f"Requested k={k} neighbors, but only {n_cells} cells are available.")

# Build a k-d tree for efficient neighbor queries
kdt = cKDTree(coords)

# Query k nearest neighbors (including self as the first neighbor)
dists, idx = kdt.query(coords, k=k)

# Compute a simple local density metric as inverse of mean distance to neighbors 2..k
# (exclude self at index 0 where distance is 0)
mean_neighbor_dist = dists[:, 1:].mean(axis=1)
local_density = 1.0 / (mean_neighbor_dist + 1e-8)

# Store local density and parameters in adata
adata.obs['local_density_k15'] = local_density
adata.uns.setdefault('local_density_params', {})['k'] = k
adata.uns['local_density_indices_k15'] = idx  # store neighbor indices for later neighborhood-composition analyses

# Check required metadata for summaries
for col in ['Populations', 'Sample_ID']:
    if col not in adata.obs.columns:
        raise KeyError(f"Expected '{col}' in adata.obs but did not find it.")

# Per-population, per-sample summary of local density
summary = (
    adata.obs
    .groupby(['Populations', 'Sample_ID'])['local_density_k15']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max'])
    .reset_index()
)

# Per-population coefficient of variation of local density across all cells
pop_stats = (
    adata.obs
    .groupby('Populations')['local_density_k15']
    .agg(['count', 'mean', 'std'])
)
pop_stats['cv'] = pop_stats['std'] / (pop_stats['mean'] + 1e-8)

# Rank populations by coefficient of variation as a proxy for spatial structure
pop_stats_sorted = pop_stats.sort_values('cv', ascending=False)

# Store summaries for downstream, and print concise overviews
adata.uns['local_density_pop_stats'] = pop_stats_sorted
adata.uns['local_density_summary_by_pop_sample'] = summary

print("Per-population local density variability (top 20 by CV):")
print(pop_stats_sorted.head(20))

print("\nPer-population, per-sample summary of local density (first 30 rows):")
print(summary.head(30))

# Compute per-cell neighborhood composition (fraction of each Population label among k-1 non-self neighbors)
pop_labels = adata.obs['Populations'].values.astype(str)
unique_pops = np.unique(pop_labels)

# Exclude self (first neighbor) when computing composition
neighbor_indices = idx[:, 1:]
neighbor_pops = pop_labels[neighbor_indices]

# For efficiency, build a (n_cells, n_pops) matrix of counts
n_pops = unique_pops.shape[0]
comp_counts = np.zeros((n_cells, n_pops), dtype=np.int32)

pop_to_col = {p: i for i, p in enumerate(unique_pops)}
for i in range(n_cells):
    # Count neighbor population labels for cell i
    vals, counts = np.unique(neighbor_pops[i], return_counts=True)
    for v, c in zip(vals, counts):
        comp_counts[i, pop_to_col[v]] = c

# Convert counts to fractions
comp_fractions = comp_counts / float(k - 1)

# Store neighborhood-composition features in adata.obs as separate columns
for j, p in enumerate(unique_pops):
    colname = f'neighbor_frac_{p}_k15'
    adata.obs[colname] = comp_fractions[:, j]

print("\nStored per-cell neighborhood composition fractions in adata.obs (columns starting with 'neighbor_frac_').")

Per-population local density variability (top 20 by CV):
             count      mean       std        cv
Populations                                     
PU            2356  0.040484  0.010866  0.268412
PX            1562  0.041512  0.010567  0.254549
PT            3726  0.043738  0.010310  0.235710
PZ            1286  0.048698  0.010973  0.225319
PS            4599  0.042237  0.009207  0.217981
PH           10887  0.043084  0.009180  0.213072
PM            7417  0.047431  0.010079  0.212495
PY            1292  0.047774  0.009540  0.199684
PAA           1027  0.052417  0.010291  0.196331
PK            8540  0.050739  0.009872  0.194568
PI           10441  0.051557  0.009486  0.183985
PG           11596  0.051381  0.009294  0.180881
PR            4673  0.051693  0.009267  0.179275
PO            5845  0.050120  0.008829  0.176149
PF           12248  0.050857  0.008623  0.169551
PL            8052  0.047096  0.007833  0.166320
PD           16624  0.050322  0.008338  0.165690
PQ          


Stored per-cell neighborhood composition fractions in adata.obs (columns starting with 'neighbor_frac_').


### Agent Interpretation

The current step is well implemented and gives you a solid foundation for the rest of the analysis plan. A few key points and concrete next steps:

1. **Populations with strongest spatial density structure**

   Using CV of `local_density_k15` as a proxy for spatial structure, you already have a ranked list. The top few:

   - PU (cv ≈ 0.27, n=2356)
   - PX (cv ≈ 0.25, n=1562)
   - PT (cv ≈ 0.24, n=3726)
   - PZ (cv ≈ 0.23, n=1286)
   - PS, PH, PM, etc. (cv ≈ 0.21–0.22)

   These are your primary candidates for “microenvironmentally structured” populations. Importantly, most have reasonable cell numbers, so they’re good choices for downstream per-population modeling.

   Feedback for later steps:
   - Focus subsequent density–complexity / density–purity analyses primarily on these top-CV populations (e.g. top 8–10), rather than all populations, to keep the testing burden manageable and tightly aligned with the hypothesis.
   - Also keep an eye on large populations with moderate CV where you expect biologically continuous tissues (e.g. PB, PC, PD, PF, PG, PI). Even if CV is lower, broad coverage may reveal subtle but robust density–complexity associations.

2. **Per-sample heterogeneity looks reasonable and exploitable**

   The per-population/per-sample summaries show:
   - Means ~0.04–0.055 and SD ~0.007–0.011 across many populations.
   - Some sample-specific variation (e.g. PH: mean density 0.045 vs 0.041 vs 0.044 across samples; PI: 0.051, 0.054, 0.048 across samples).

   This supports the plan to:
   - Include sample as a covariate (or random effect) in your density–complexity / density–purity models.
   - Also consider sample-stratified descriptive plots to distinguish true microenvironment effects from section-level architecture.

3. **k and density metric choices**

   Using k=15 and inverse mean distance to neighbors 2..15 is a reasonable compromise between locality and robustness.

   For later steps, I’d suggest:
   - Do not change k mid-analysis; downstream neighborhood composition (fractions) and density–expression associations should all use the same k=15 to maintain interpretability.
   - Optionally, you can run a minimal sensitivity check later (e.g. k=10, 20) for one or two populations to confirm qualitative robustness, but keep the primary analysis on k=15 to avoid multiplying tests.

4. **Neighborhood composition features are ready and flexible**

   The `neighbor_frac_*_k15` columns in `.obs` are exactly what you need for:
   - Testing whether a focal population’s complexity/purity associates with specific neighbor types (plan steps 3–4).
   - Later per-gene analyses that incorporate either density or key neighbor fractions.

   One technical concern:
   - You’re using `np.unique` per cell to count neighbor populations, which is fine at current scale but O(n_cells × k × log k). If performance becomes an issue in later steps when layering additional per-cell operations, consider vectorized counting or sparse representations. For now it’s acceptable.

5. **How to move into step 2 (density–complexity / density–purity associations)**

   Concrete suggestions for the next step, tailored to what you already computed:

   - Within each candidate population (e.g. PU, PX, PT, PZ, PS, PH, PM, PAA, PK):
     - Extract:
       - `local_density_k15`
       - A measure of transcriptional complexity (e.g. `n_genes_by_counts` or similar)
       - UMI count (`total_counts`)
       - Purity metric you’ve previously defined (or construct one now: e.g., 1 – max(other-population signature, or contamination proxy); be explicit).
       - `Sample_ID`.

   - Fit per-population models such as:
     - Complexity ~ local_density_k15 + C(Sample_ID)
     - Purity ~ local_density_k15 + C(Sample_ID)
     - Optionally include `log10(total_counts)` to separate density from sequencing depth:
       - Complexity ~ local_density_k15 + log10(UMI) + C(Sample_ID)
       - Purity ~ local_density_k15 + log10(UMI) + C(Sample_ID)

   - Compute:
     - Effect size (slope of density).
     - p-value for density term.
     - Partial R² or similar to quantify density’s added explanatory power beyond sample and depth.

   This directly tests the central part of your hypothesis: whether local density relates systematically to complexity/purity in specific populations.

6. **Preparing for dense vs sparse comparisons (step 3)**

   Your density distribution appears relatively narrow but with meaningful variation (e.g. PH SD ~0.009 on mean ~0.043). This should be enough to define quantile-based strata:

   - Within each focal population, define:
     - “Sparse” = bottom 20% (or 25%) of `local_density_k15`.
     - “Dense” = top 20% (or 25%).
   - Within each sample, you might want to define quantiles sample-wise to avoid conflating sample density shifts with true local differences.

   - Then:
     - Compare complexity and purity between dense vs sparse via Wilcoxon rank-sum (within population, optionally controlling for sample by either stratifying or including sample as a blocking factor via permutation).

   This will give you interpretable contrasts like “PU cells in dense microenvironments show +X% higher median complexity and +Y difference in purity than PU cells in sparse microenvironments.”

7. **How to exploit neighborhood composition (steps 4–5)**

   With `neighbor_frac_*_k15` already computed:

   - For each focal population (e.g. PU), define:
     - Predictor matrix: [local_density_k15, neighbor_frac_PA_k15, neighbor_frac_PB_k15, ...] but to avoid collinearity, consider:
       - Exclude the focal population’s own neighbor fraction (or use it plus a reduced set of neighbors).
       - Or use a smaller, biologically/empirically selected set of neighbor types that actually appear around the focal population (e.g. neighbors with mean fraction > 0.05 around that population).

   - Start with global phenotype models:
     - Complexity ~ local_density_k15 + neighbor_frac_PX_k15 + neighbor_frac_PY_k15 + ... + C(Sample_ID)
     - Purity ~ same predictors.

   - This lets you test:
     - Which neighbor types impact complexity/purity after accounting for overall density.
     - Whether, for example, PU cells surrounded by high PX fraction have higher complexity even at similar density.

   - For gene-level associations (step 5):
     - Within each focal population, per gene, compute Spearman ρ with:
       - `local_density_k15` (primary microenvironmental axis).
       - Selected neighbor fractions that showed strong effects in the phenotype models.
     - Apply FDR correction per population.
     - Summarize: top 20 positively and top 20 negatively associated genes for density and for key neighbor fractions.

   This is likely where your analysis will become clearly distinct from past gradient/signature work: you’re explicitly leveraging discrete neighborhood composition alongside local density.

8. **Integration / ranking of microenvironment sensitivity (step 6)**

   You already have per-population CV in density (`pop_stats_sorted`). To build a final ranking table that addresses the hypothesis, consider aggregating:

   - From step 2:
     - Density–complexity slope + p-value per population.
     - Density–purity slope + p-value.
   - From step 3:
     - Dense vs sparse effect sizes on complexity and purity (e.g. median difference or log fold change), plus test p-values.
   - From step 4:
     - Number of neighbor-type fractions with significant association to complexity/purity after controlling for density.
   - From step 5:
     - # of genes with FDR<0.05 associated with density.
     - # of genes with FDR<0.05 associated with any key neighbor fraction.

   Combine these with your current CV metric to produce a table like:

   - Population | n_cells | density_cv | density–complexity_slope | density–complexity_p | density–purity_slope | density–purity_p | dense_vs_sparse_complexity_effect | dense_vs_sparse_purity_effect | #density_associated_genes | #neighbor_associated_genes | key_neighbor_types.

   Populations that score highly across multiple columns would be your prime candidates supporting the hypothesis (microenvironment-sensitive maturation or stress states).

9. **On hypothesis validation so far**

   At this stage, you’ve only established:
   - There is substantial variation in local density within most populations (CV up to ~0.27), and
   - You can quantify each cell’s neighborhood composition.

   This supports the *premise* of the hypothesis (that there is microenvironmental heterogeneity within populations) but does not yet show a link to transcriptional complexity or purity. The next steps (density–complexity / purity modeling, dense vs sparse contrasts, and neighborhood composition analyses) are essential to either validate or refute the hypothesis.

In summary: the infrastructure is in place and looks technically sound. Focus next on the high-CV populations (PU, PX, PT, PZ, PS, PH, PM, PAA, PK) for density–complexity/purity models with sample adjustment, followed by dense vs sparse contrasts and neighbor-fraction modeling, then finally gene-level correlations. That will give you a coherent, distinct, and biologically interpretable test of your microenvironmental hypothesis.

## Next Steps
Step 1: Within each sufficiently large and spatially structured population (e.g., top 8–10 by coefficient of variation of local_density_k15), quantify the association between local spatial density and transcriptional complexity, UMI count, and purity using per-population correlation and simple regression models that include Sample_ID as a fixed-effect covariate, and summarize effect sizes and (raw and FDR-adjusted) p-values.
Step 2: For key populations showing significant density–complexity or density–purity associations, compare complexity and purity between cells in dense versus sparse microenvironments (e.g., lowest vs highest density quintiles within each sample) using non-parametric tests, and report standardized effect sizes and adjusted p-values.
Step 3: For the same key populations, model how immediate neighborhood composition (selected neighbor_frac_*_k15 features with non-trivial prevalence) relates to complexity and purity after controlling for local density and Sample_ID, using multiple linear regression per population, and summarize which neighbor types have significant positive or negative associations.
Step 4: Within each focal population, identify genes whose expression is significantly associated with local density using per-gene Spearman correlations and Benjamini–Hochberg FDR correction, and report counts of density-associated genes and representative top hits.
Step 5: Integrate population-level metrics (density CV, density–complexity/purity slopes and p-values, dense vs sparse contrasts, significant neighbor-composition effects, and counts of density-associated genes) into a single summary table ranking populations by microenvironmental sensitivity, ensuring all outputs remain text-only without new embeddings or gradient visualizations.

## This code implements step 1 of the updated plan by selecting the top spatially structured populations, then quantifying per-population associations between local density and complexity/UMI/purity while treating Sample_ID as a fixed effect and controlling for log-transformed UMI in partial correlations, with robust correlation safeguards and Benjamini–Hochberg FDR correction across populations.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Ensure required columns are present
required_obs_cols = ['Populations', 'Sample_ID', 'UMI Count', 'Complexity', 'Purity', 'local_density_k15']
missing = [c for c in required_obs_cols if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing required columns in adata.obs: {missing}")

# Prepare a working DataFrame with needed variables
obs_df = adata.obs.copy()
obs_df['Populations'] = obs_df['Populations'].astype(str)
obs_df['Sample_ID'] = obs_df['Sample_ID'].astype(str)

# Define candidate populations: sufficiently large and with stored CV (already computed in adata.uns['local_density_pop_stats'])
if 'local_density_pop_stats' not in adata.uns:
    raise KeyError("Expected 'local_density_pop_stats' in adata.uns from the previous step.")

pop_stats_df = adata.uns['local_density_pop_stats'].copy()
if not isinstance(pop_stats_df, pd.DataFrame):
    pop_stats_df = pd.DataFrame(pop_stats_df)

# Ensure expected columns exist
for col in ['cv', 'count']:
    if col not in pop_stats_df.columns:
        raise KeyError(f"Expected column '{col}' in adata.uns['local_density_pop_stats'].")

# Sort by CV descending
pop_stats_df = pop_stats_df.sort_values('cv', ascending=False)

# Filter to populations with at least a minimal number of cells (e.g., >= 300) for stable estimates
min_cells = 300
eligible = pop_stats_df[pop_stats_df['count'] >= min_cells]

# Optionally restrict to top N by CV to stay focused on spatially structured populations
top_n = 10
candidate_pops = eligible.index.tolist()[:top_n]

print("Candidate populations for density–complexity/purity modeling (ordered by CV, with counts):")
print(eligible.loc[candidate_pops, ['count', 'cv']])

results = []

# Helper to safely compute Pearson correlation (returns nan if any vector is constant or too small)
def safe_pearsonr(x, y):
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3:
        return np.nan, np.nan
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan
    r, p = stats.pearsonr(x, y)
    return r, p

# Helper to compute partial correlation between x and y while controlling for covariates (linear residualization)
def partial_corr(x, y, covariates):
    """Compute partial correlation between x and y controlling for covariates using linear residualization.
    x, y: 1D numpy arrays
    covariates: 2D array (n_samples, n_covariates)
    Returns (r, p) from Pearson correlation between residuals.
    """
    import numpy.linalg as LA

    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(covariates).all(axis=1)
    x = x[mask]
    y = y[mask]
    C = covariates[mask]

    if x.size < 10:
        return np.nan, np.nan

    # Add intercept
    C_design = np.column_stack([np.ones(C.shape[0]), C])

    # Solve least-squares for x and y
    beta_x, _, _, _ = LA.lstsq(C_design, x, rcond=None)
    beta_y, _, _, _ = LA.lstsq(C_design, y, rcond=None)

    x_res = x - C_design.dot(beta_x)
    y_res = y - C_design.dot(beta_y)

    # Guard against constant residuals
    if np.std(x_res) == 0 or np.std(y_res) == 0:
        return np.nan, np.nan

    r, p = stats.pearsonr(x_res, y_res)
    return r, p

# Encode Sample_ID as fixed effects via dummy variables for partial correlation (covariates)
sample_dummies_global = pd.get_dummies(obs_df['Sample_ID'], drop_first=True)

for pop in candidate_pops:
    sub = obs_df[obs_df['Populations'] == pop].copy()
    n_cells_pop = sub.shape[0]
    if n_cells_pop < min_cells:
        continue

    print(f"\nAnalyzing population {pop} (n={n_cells_pop})")

    # Extract variables
    density = sub['local_density_k15'].values.astype(float)
    umi = sub['UMI Count'].values.astype(float)
    complexity = sub['Complexity'].values.astype(float)
    purity = sub['Purity'].values.astype(float)

    # Construct covariate matrix: log10(UMI) and Sample_ID fixed effects (dummy encoding)
    log_umi = np.log10(umi + 1.0)
    sub_samples = sub['Sample_ID']
    sub_dummies = pd.get_dummies(sub_samples, drop_first=True)
    # Ensure consistent columns across populations
    sub_dummies = sub_dummies.reindex(columns=sample_dummies_global.columns, fill_value=0)
    covar_mat = np.column_stack([log_umi, sub_dummies.values])

    # 1) Simple Pearson correlation between density and each outcome (unadjusted)
    r_complexity, p_complexity = safe_pearsonr(density, complexity)
    r_umi, p_umi = safe_pearsonr(density, umi)
    r_purity, p_purity = safe_pearsonr(density, purity)

    # 2) Partial correlations controlling for log10(UMI) and Sample_ID (fixed effects)
    pr_complexity, pp_complexity = partial_corr(density, complexity, covar_mat)
    pr_purity, pp_purity = partial_corr(density, purity, covar_mat)

    # 3) Simple linear regression slopes (univariate density effect) for interpretability
    # complexity ~ a + b * density
    slope_c, intercept_c, r_c, p_c, stderr_c = stats.linregress(density, complexity)
    # purity ~ a + b * density
    slope_p, intercept_p, r_p, p_p, stderr_p = stats.linregress(density, purity)

    # Per-population means and SDs for interpretability
    mean_density, sd_density = float(np.mean(density)), float(np.std(density))
    mean_complexity, sd_complexity = float(np.mean(complexity)), float(np.std(complexity))
    mean_umi, sd_umi = float(np.mean(umi)), float(np.std(umi))
    mean_purity, sd_purity = float(np.mean(purity)), float(np.std(purity))

    result_row = {
        'Population': pop,
        'n_cells': n_cells_pop,
        'mean_density_k15': mean_density,
        'sd_density_k15': sd_density,
        'mean_complexity': mean_complexity,
        'sd_complexity': sd_complexity,
        'mean_umi': mean_umi,
        'sd_umi': sd_umi,
        'mean_purity': mean_purity,
        'sd_purity': sd_purity,
        'pearson_r_density_complexity': r_complexity,
        'pearson_raw_p_density_complexity': p_complexity,
        'pearson_r_density_umi': r_umi,
        'pearson_raw_p_density_umi': p_umi,
        'pearson_r_density_purity': r_purity,
        'pearson_raw_p_density_purity': p_purity,
        'partial_r_density_complexity_adj_logumi_sampleFE': pr_complexity,
        'partial_raw_p_density_complexity_adj_logumi_sampleFE': pp_complexity,
        'partial_r_density_purity_adj_logumi_sampleFE': pr_purity,
        'partial_raw_p_density_purity_adj_logumi_sampleFE': pp_purity,
        'slope_complexity_vs_density': slope_c,
        'slope_stderr_complexity_vs_density': stderr_c,
        'raw_p_slope_complexity_vs_density': p_c,
        'slope_purity_vs_density': slope_p,
        'slope_stderr_purity_vs_density': stderr_p,
        'raw_p_slope_purity_vs_density': p_p
    }
    results.append(result_row)

# Compile results into a DataFrame
results_df = pd.DataFrame(results)

# Apply Benjamini–Hochberg FDR correction across populations for key partial-correlation p-values (complexity and purity)
if not results_df.empty:
    def bh_fdr(pvals):
        pvals = np.asarray(pvals, dtype=float)
        n = pvals.size
        order = np.argsort(pvals)
        ranked_pvals = pvals[order]
        qvals = np.empty(n, dtype=float)
        prev_q = 1.0
        for i in range(n - 1, -1, -1):
            rank = i + 1
            q = ranked_pvals[i] * n / rank
            if q > prev_q:
                q = prev_q
            prev_q = q
            qvals[i] = q
        # Reorder back to original
        qvals_corrected = np.empty(n, dtype=float)
        qvals_corrected[order] = qvals
        return qvals_corrected

    results_df['partial_fdr_p_density_complexity_adj_logumi_sampleFE'] = bh_fdr(results_df['partial_raw_p_density_complexity_adj_logumi_sampleFE'].values)
    results_df['partial_fdr_p_density_purity_adj_logumi_sampleFE'] = bh_fdr(results_df['partial_raw_p_density_purity_adj_logumi_sampleFE'].values)

    # Sort by FDR-adjusted partial p-value for complexity (ascending)
    results_df = results_df.sort_values('partial_fdr_p_density_complexity_adj_logumi_sampleFE')

# Store in adata.uns for downstream steps
adata.uns['density_complexity_purity_models'] = results_df

print("\nPer-population density–complexity/UMI/purity association summary (sorted by FDR-adjusted complexity partial-correlation p-value):")
if results_df.empty:
    print("No candidate populations met the criteria for analysis.")
else:
    print(results_df.to_string(index=False))


Candidate populations for density–complexity/purity modeling (ordered by CV, with counts):
             count        cv
Populations                 
PU            2356  0.268412
PX            1562  0.254549
PT            3726  0.235710
PZ            1286  0.225319
PS            4599  0.217981
PH           10887  0.213072
PM            7417  0.212495
PY            1292  0.199684
PAA           1027  0.196331
PK            8540  0.194568

Analyzing population PU (n=2356)

Analyzing population PX (n=1562)

Analyzing population PT (n=3726)

Analyzing population PZ (n=1286)

Analyzing population PS (n=4599)

Analyzing population PH (n=10887)

Analyzing population PM (n=7417)

Analyzing population PY (n=1292)

Analyzing population PAA (n=1027)

Analyzing population PK (n=8540)

Per-population density–complexity/UMI/purity association summary (sorted by FDR-adjusted complexity partial-correlation p-value):
Population  n_cells  mean_density_k15  sd_density_k15  mean_complexity  sd_complexity   

### Agent Interpretation

Several aspects of these results are very supportive of the hypothesis and give clear guidance for next steps.

1. **Evidence that local density is strongly linked to transcriptional complexity (even after adjustment)**  
   - All 10 candidate populations show **positive partial correlations** between density and complexity after adjusting for log10(UMI) and Sample_ID, and these are highly significant (FDR ≪ 0.05 in every case).  
   - Effect sizes are large for several populations:
     - PM: partial r ≈ 0.316  
     - PK: partial r ≈ 0.291  
     - PZ: partial r ≈ 0.303  
     - PX: partial r ≈ 0.294  
     - PAA/PY/PH are also ~0.20–0.28.  
   - These are not trivial correlations for single-cell data, and they’re robust to both UMI and sample effects. This directly supports the “density–complexity” component of your hypothesis.

2. **Density–UMI vs density–complexity: complexity is not just tracking UMI**  
   - In many populations, **raw Pearson r(density, UMI)** is weak or even negative (e.g., PM: r ≈ −0.11; PK: r ≈ −0.23; PX: r ≈ −0.13; PZ, PY, PAA also negative).  
   - Yet **density–complexity correlations are positive and strong** in the same populations (r ≈ 0.25–0.35).  
   - This divergence is important: local density is associated with more complex transcriptomes in a way that is *not* just “more dense → more UMIs → more genes detected.” Your explicit adjustment for log10(UMI) in the partial correlations reinforces this interpretation.

3. **Density–purity relationships are present but more heterogeneous**  
   - Several populations show **negative partial correlations** between density and purity:
     - PM: partial r ≈ −0.126 (FDR ≈ 9.5×10⁻²⁷)  
     - PX: partial r ≈ −0.168 (FDR ≈ 5.8×10⁻¹¹)  
     - PZ: partial r ≈ −0.085 (FDR ≈ 2.8×10⁻³)  
     - PAA: partial r ≈ −0.134 (FDR ≈ 2.3×10⁻⁵)  
     - PS and PY also modestly negative and significant.  
   - Others (PH, PT, PU) have small positive or near-zero partial r.  
   - This suggests that “purity” responds to density in a **population-specific way**, and not always in the same direction as complexity. That nuance is actually useful: it gives you a way to classify which populations become more “mixed” in dense microenvironments versus those that maintain or even increase purity.

4. **Populations that look especially “microenvironment-sensitive”**  
   For prioritizing subsequent steps (quintile contrasts, neighbor-composition modeling, per-gene density associations), I’d rank candidates roughly as:

   - **Tier 1 (very strong complexity–density effects, significant purity–density effects):**
     - **PM**: highest partial r for complexity (~0.316), strong negative purity association (partial r ~−0.126), very large n (7,417). Very compelling as a primary focus.
     - **PK**: partial r for complexity ~0.291, negligible purity effect (−0.01, ns), but extremely significant overall; large n (8,540). Good for examining density–complexity without confounding purity changes.
     - **PX**: partial r complexity ~0.294; strong negative partial r purity ~−0.168. This is ideal for dissecting how dense niches simultaneously increase complexity and decrease purity.
     - **PZ**: partial r complexity ~0.303; modest negative purity association.  

   - **Tier 2 (strong but slightly smaller effects or more modest purity changes):**
     - **PAA**: partial r complexity ~0.275; clear negative purity association.  
     - **PY**: partial r complexity ~0.277; weaker but significant purity effect.  
     - **PH**: partial r complexity ~0.202; positive purity association (~0.122). This makes PH a good contrast case where density increases both complexity and purity.

   - **Tier 3 (weaker complexity–density effects but still significant):**
     - **PS, PT, PU**: partial r complexity ~0.05–0.18. Still significant due to high n, but smaller in magnitude.

   For the next steps, I would prioritize **PM, PK, PX, PZ, PAA, PH** as “key populations,” with **PM and PX** as top choices because they show both strong complexity effects and clearly interpretable purity shifts.

5. **Implications for the hypothesis about maturation or stress states**  
   - Populations where **dense regions have higher complexity but lower purity** (e.g. PM, PX, PAA, PZ) are consistent with:
     - Either microenvironment-induced **state diversification** (cells in dense niches express additional modules or stress/interaction programs that broaden their transcriptomes while also picking up more “impure” signatures), or  
     - **Increased local mixing/interaction** at dense interfaces, which might result in expression of signals characteristic of neighboring types (reducing purity but increasing the number of genes detected).
   - Populations where **density increases both complexity and purity** (e.g. PH) could reflect **mature, specialized clusters** where dense packing goes along with a more defined, high-complexity program but still relatively type-specific.

   These patterns are very much in line with your idea that microenvironmental context (local density and neighborhood) modulates maturation or stress programs, and they extend beyond simple technical explanations.

6. **Concrete suggestions for upcoming planned steps**

   **Step 2: Dense vs sparse comparisons (quintiles)**  
   - Do this per sample within the key populations to respect your design (“lowest vs highest density quintiles within each sample”).
   - Focus first on **PM, PX, PZ, PAA, PH, PK**:
     - In each, compare:
       - **Complexity** dense vs sparse: expect large standardized differences, especially in PM, PX, PK, PZ.  
       - **Purity** dense vs sparse: for PM/PX/PAA/PZ, expect lower purity in dense quintiles; for PH maybe higher purity.  
     - Use **effect sizes (Cohen’s d or rank-biserial)** plus FDR across pops and outcomes; you already have correlation evidence, but showing discrete contrasts will be intuitive and robust to non-linearity.
   - Check whether the dense vs sparse contrasts are **consistent across samples** (e.g., via per-sample effects and then meta-analysis or a mixed-effects view) to ensure the patterns are not dominated by a single section.

   **Step 3: Neighborhood composition modeling (beyond density)**  
   - For the same key populations, fit regression models:
     - outcome ~ local_density_k15 + selected neighbor_frac_*_k15 + Sample_ID (FE)  
   - Given your results:
     - For **PM and PX**, look for neighbor types whose higher local fraction is associated with **increased complexity but decreased purity**, which would strongly support context-driven modulation (e.g., specific immune-like or signaling-rich neighbors).
     - For **PH/PK**, you might find neighbor types that reinforce identity and purity in dense regions.
   - Be conservative in selecting neighbor fractions: include only those with non-trivial prevalence to avoid unstable estimates.

   **Step 4: Per-gene associations with density**  
   - For the most microenvironment-sensitive populations (PM, PX, PZ, PAA, PH, PK), compute per-gene Spearman correlations with density:
     - You may also want to **partial out log10(UMI) and Sample_ID** (either via residualization or modeling), mirroring what you did for complexity. Otherwise, at least stratify or regress out UMI beforehand.
   - Summarize:
     - Number of **density-upregulated vs density-downregulated** genes per population.  
     - Overlap of “density genes” across populations: shared vs population-specific density signatures. This will help you distinguish generic stress programs from population-specific maturation programs.
   - These density-associated genes will be crucial later when interpreting whether the density–complexity relationship reflects **stress responses (e.g., unfolded protein response, hypoxia-like genes)** versus **developmental maturation/signaling**.

   **Step 5: Integrated summary and ranking**  
   - Given the strong partial correlations you already have, the integrated ranking will likely place **PM, PX, PZ, PK, PAA, PH** at the top.  
   - You can use:
     - Density CV (already high in these),
     - Magnitude and FDR of partial r(density, complexity),
     - Strength and direction of purity effects,
     - Dense vs sparse contrasts,
     - Number and effect sizes of significant neighbor-composition coefficients,
     - Counts of density-associated genes.

7. **Checks and possible refinements to current step**

   - The current approach **encodes Sample_ID as fixed effects** and adjusts for log10(UMI); this closely matches the hypothesis and is statistically sound for large n.  
   - A few optional refinements for robustness—useful if you want to be exceedingly cautious:
     - Verify that complexity and purity distributions are not strongly non-linear with density (e.g., check whether slopes are similar if you model log-density or use splines).  
     - Consider also computing **Spearman partial correlations** (e.g., via residuals) to reduce sensitivity to outliers, although your large n and small p-values suggest the patterns are strong.

Overall, the current step provides strong quantitative evidence that **local cell density is robustly associated with transcriptional complexity across multiple spatially structured cardiac populations, beyond sample and UMI effects.** Purity shows more heterogeneous, population-specific associations with density, which is actually informative and will help you classify microenvironmental responses.  

The next steps should zoom in on PM, PX, PZ, PAA, PH, and PK, using density-stratified contrasts, neighborhood-composition modeling, and per-gene density associations to determine whether these patterns reflect maturation gradients, stress programs, or distinct interaction niches.

## Next Steps
Step 1: For key microenvironment-sensitive populations (PM, PX, PZ, PAA, PH, PK) identified in prior density–complexity modeling, compare transcriptional complexity and purity between dense and sparse microenvironments defined within each sample using local_density_k15–based quantiles, quantifying per-population effect sizes and statistical significance with non-parametric tests and FDR correction.
Step 2: Summarize dense–sparse contrasts for these key populations in a consolidated table that includes direction and magnitude of dense–sparse differences for complexity and purity, their adjusted p-values, and a simple classification of populations into density-associated increase vs decrease in purity to prioritize microenvironment-sensitive populations.
Step 3: For the same key populations, model how immediate neighborhood composition (selected neighbor_frac_*_k15 features with non-trivial prevalence) relates to complexity and purity after controlling for local_density_k15, log10(UMI Count), and Sample_ID fixed effects, and report which neighbor types show robust positive or negative associations.
Step 4: Within each key population, identify genes whose expression is significantly associated with local_density_k15 after regressing out log10(UMI Count) and Sample_ID (and optionally key neighbor fractions), using per-gene Spearman correlations and Benjamini–Hochberg FDR correction, and compile a text-only summary of density-associated genes (counts and representative top hits) to distinguish putative maturation versus stress programs.

## This code implements the dense-versus-sparse microenvironment comparison for predefined key cardiac populations by defining sample-wise density strata from local_density_k15, then testing dense vs sparse differences in transcriptional complexity and purity with Mann–Whitney U tests, while storing effect sizes, FDR-corrected p-values, and reusable per-cell density-stratum labels for downstream summarization and modeling.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Ensure required columns are present and properly typed
required_cols = ['Populations', 'Sample_ID', 'UMI Count', 'Complexity', 'Purity', 'local_density_k15']
missing = [c for c in required_cols if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing required columns in adata.obs: {missing}")

obs = adata.obs.copy()
obs['Populations'] = obs['Populations'].astype(str)
obs['Sample_ID'] = obs['Sample_ID'].astype(str)
for col in ['Complexity', 'Purity', 'local_density_k15', 'UMI Count']:
    obs[col] = obs[col].astype(float)

# Use predefined key microenvironment-sensitive populations
key_pops = ['PM', 'PX', 'PZ', 'PAA', 'PH', 'PK']
key_pops = [p for p in key_pops if p in obs['Populations'].unique()]
if len(key_pops) == 0:
    raise ValueError("None of the predefined key populations (PM, PX, PZ, PAA, PH, PK) are present in adata.obs['Populations'].")

print("Key populations for dense vs sparse comparison:", key_pops)

# Parameters for density strata
low_q = 0.2  # bottom 20% = sparse
high_q = 0.8 # top 20% = dense
min_cells_stratum = 30  # require at least this many cells per stratum per population

rows = []

# Initialize a global density-stratum column so downstream steps can reuse it
if 'density_stratum_k15' not in obs.columns:
    obs['density_stratum_k15'] = 'mid'

for pop in key_pops:
    sub_pop = obs[obs['Populations'] == pop].copy()
    if sub_pop.shape[0] < 2 * min_cells_stratum:
        print(f"Skipping population {pop}: not enough cells for robust strata (n={sub_pop.shape[0]}).")
        continue

    print(f"\nProcessing population {pop} (n={sub_pop.shape[0]})")

    # Define dense/sparse labels within each sample using sample-wise quantiles
    strata = []
    for sid, sub_s in sub_pop.groupby('Sample_ID'):
        if sub_s.shape[0] < 2 * min_cells_stratum:
            # Mark these cells as mid; they will be ignored in dense/sparse contrasts
            sub_s = sub_s.copy()
            sub_s['density_stratum'] = 'mid'
            strata.append(sub_s)
            continue
        dens = sub_s['local_density_k15'].values.astype(float)
        q_low = np.quantile(dens, low_q)
        q_high = np.quantile(dens, high_q)

        labels = np.full(sub_s.shape[0], 'mid', dtype=object)
        labels[dens < q_low] = 'sparse'
        labels[dens > q_high] = 'dense'

        sub_s = sub_s.copy()
        sub_s['density_stratum'] = labels
        strata.append(sub_s)

    sub_pop_q = pd.concat(strata, axis=0)

    # Update global density_stratum_k15 labels for this population's cells
    obs.loc[sub_pop_q.index, 'density_stratum_k15'] = sub_pop_q['density_stratum']

    # Restrict to dense and sparse across all samples
    mask_ds = sub_pop_q['density_stratum'].isin(['dense', 'sparse'])
    sub_ds = sub_pop_q[mask_ds].copy()

    n_dense = (sub_ds['density_stratum'] == 'dense').sum()
    n_sparse = (sub_ds['density_stratum'] == 'sparse').sum()

    print(f"  Dense cells: {n_dense}, Sparse cells: {n_sparse}")

    if n_dense < min_cells_stratum or n_sparse < min_cells_stratum:
        print(f"  Skipping Wilcoxon tests for {pop}: insufficient dense/sparse cells after sample-wise stratification.")
        continue

    # Extract variables
    is_dense = (sub_ds['density_stratum'] == 'dense').values
    complexity_dense = sub_ds.loc[is_dense, 'Complexity'].values
    complexity_sparse = sub_ds.loc[~is_dense, 'Complexity'].values
    purity_dense = sub_ds.loc[is_dense, 'Purity'].values
    purity_sparse = sub_ds.loc[~is_dense, 'Purity'].values

    # Non-parametric comparisons: two-sided Mann–Whitney U (Wilcoxon rank-sum)
    u_comp, p_comp = stats.mannwhitneyu(complexity_dense, complexity_sparse, alternative='two-sided')
    u_purity, p_purity = stats.mannwhitneyu(purity_dense, purity_sparse, alternative='two-sided')

    # Effect size: rank-biserial correlation for Mann–Whitney U (dense minus sparse)
    n1 = complexity_dense.size
    n2 = complexity_sparse.size
    rb_comp = 1.0 - (2.0 * u_comp) / (n1 * n2)

    n1_p = purity_dense.size
    n2_p = purity_sparse.size
    rb_purity = 1.0 - (2.0 * u_purity) / (n1_p * n2_p)

    # Also report mean, median, and explicit dense–sparse differences
    mean_comp_dense = float(np.mean(complexity_dense))
    mean_comp_sparse = float(np.mean(complexity_sparse))
    med_comp_dense = float(np.median(complexity_dense))
    med_comp_sparse = float(np.median(complexity_sparse))

    mean_purity_dense = float(np.mean(purity_dense))
    mean_purity_sparse = float(np.mean(purity_sparse))
    med_purity_dense = float(np.median(purity_dense))
    med_purity_sparse = float(np.median(purity_sparse))

    rows.append({
        'Population': pop,
        'n_total_pop': int(sub_pop.shape[0]),
        'n_dense': int(n_dense),
        'n_sparse': int(n_sparse),
        'mean_complexity_dense': mean_comp_dense,
        'mean_complexity_sparse': mean_comp_sparse,
        'delta_mean_complexity_dense_minus_sparse': mean_comp_dense - mean_comp_sparse,
        'median_complexity_dense': med_comp_dense,
        'median_complexity_sparse': med_comp_sparse,
        'delta_median_complexity_dense_minus_sparse': med_comp_dense - med_comp_sparse,
        'mean_purity_dense': mean_purity_dense,
        'mean_purity_sparse': mean_purity_sparse,
        'delta_mean_purity_dense_minus_sparse': mean_purity_dense - mean_purity_sparse,
        'median_purity_dense': med_purity_dense,
        'median_purity_sparse': med_purity_sparse,
        'delta_median_purity_dense_minus_sparse': med_purity_dense - med_purity_sparse,
        'mw_U_complexity': float(u_comp),
        'raw_p_complexity_dense_vs_sparse': float(p_comp),
        'rank_biserial_complexity_dense_vs_sparse': float(rb_comp),
        'mw_U_purity': float(u_purity),
        'raw_p_purity_dense_vs_sparse': float(p_purity),
        'rank_biserial_purity_dense_vs_sparse': float(rb_purity)
    })

# Compile results and apply BH-FDR across populations for both outcomes
if len(rows) == 0:
    print("\nNo populations had sufficient dense and sparse cells for comparison.")
    dense_sparse_df = pd.DataFrame()
else:
    dense_sparse_df = pd.DataFrame(rows)

    def bh_fdr(pvals):
        pvals = np.asarray(pvals, dtype=float)
        n = pvals.size
        order = np.argsort(pvals)
        ranked = pvals[order]
        qvals = np.empty(n, dtype=float)
        prev_q = 1.0
        for i in range(n - 1, -1, -1):
            rank = i + 1
            q = ranked[i] * n / rank
            if q > prev_q:
                q = prev_q
            prev_q = q
            qvals[i] = q
        out = np.empty(n, dtype=float)
        out[order] = qvals
        return out

    if 'raw_p_complexity_dense_vs_sparse' in dense_sparse_df.columns and dense_sparse_df.shape[0] > 0:
        dense_sparse_df['fdr_p_complexity_dense_vs_sparse'] = bh_fdr(dense_sparse_df['raw_p_complexity_dense_vs_sparse'].values)
    if 'raw_p_purity_dense_vs_sparse' in dense_sparse_df.columns and dense_sparse_df.shape[0] > 0:
        dense_sparse_df['fdr_p_purity_dense_vs_sparse'] = bh_fdr(dense_sparse_df['raw_p_purity_dense_vs_sparse'].values)

    # Sort primarily by adjusted p-value for complexity, then by absolute rank-biserial effect size
    if 'fdr_p_complexity_dense_vs_sparse' in dense_sparse_df.columns:
        dense_sparse_df = dense_sparse_df.sort_values(
            ['fdr_p_complexity_dense_vs_sparse', 'rank_biserial_complexity_dense_vs_sparse'],
            ascending=[True, False]
        )

# Store in adata.uns for downstream steps and push the global labels back into adata.obs
adata.uns['dense_sparse_density_complexity_purity'] = dense_sparse_df
adata.obs['density_stratum_k15'] = obs['density_stratum_k15']

print("\nDense vs sparse microenvironment comparison (per population):")
if dense_sparse_df.empty:
    print("No results (insufficient dense/sparse cells).")
else:
    # Round for readability in text-only output
    print(dense_sparse_df.round(4).to_string(index=False))

Key populations for dense vs sparse comparison: ['PM', 'PX', 'PZ', 'PAA', 'PH', 'PK']

Processing population PM (n=7417)
  Dense cells: 1484, Sparse cells: 1484

Processing population PX (n=1562)
  Dense cells: 313, Sparse cells: 313

Processing population PZ (n=1286)
  Dense cells: 257, Sparse cells: 257

Processing population PAA (n=1027)
  Dense cells: 207, Sparse cells: 207

Processing population PH (n=10887)
  Dense cells: 2179, Sparse cells: 2179

Processing population PK (n=8540)
  Dense cells: 1708, Sparse cells: 1708

Dense vs sparse microenvironment comparison (per population):
Population  n_total_pop  n_dense  n_sparse  mean_complexity_dense  mean_complexity_sparse  delta_mean_complexity_dense_minus_sparse  median_complexity_dense  median_complexity_sparse  delta_median_complexity_dense_minus_sparse  mean_purity_dense  mean_purity_sparse  delta_mean_purity_dense_minus_sparse  median_purity_dense  median_purity_sparse  delta_median_purity_dense_minus_sparse  mw_U_complexity  

### Agent Interpretation

These results are very supportive of the core hypothesis and give you a clear set of populations and directions to pursue in the next steps.

Key points from the dense–sparse contrasts:

1. **Transcriptional complexity: strong, consistent density effect across all key populations**

- In every key population (PM, PX, PZ, PAA, PH, PK), **dense cells have substantially higher complexity** than sparse cells:
  - Δ(mean complexity dense – sparse) ranges from ~0.97 (PH) up to ~3 (PZ, PAA, PM, PK).
  - Medians shift in the same direction (e.g., PM: 10 vs 6; PZ: 11 vs 8; PAA: 13 vs 10).
- Mann–Whitney tests for complexity are **highly significant** (raw p = 0 for all; FDR = 0 for all).
- Rank-biserial correlations for complexity are **moderate-to-strong in magnitude** and consistently **negative** for the reported ordering:
  - PM: -0.49, PZ: -0.55, PX: -0.46, PAA: -0.40, PK: -0.48, PH: -0.27
  - The sign just reflects the specific “group1 vs group2” ordering in the Mann–Whitney; the direction from the raw means/medians is clear: **dense > sparse in complexity**.

**Interpretation for the hypothesis:**  
This strongly supports the idea that for these microenvironment-sensitive populations, local density is coupled to increased transcriptional complexity beyond UMI and sample, at least at the marginal (unadjusted) level. The effect is large and consistent enough that it almost certainly persists when you later control for covariates, though that still needs formal modeling.

2. **Purity: directionally coherent, but weaker and heterogeneous**

Purity behaves differently and more subtly:

- **Most populations show lower purity in dense microenvironments:**
  - PM: mean purity dense 0.537 vs sparse 0.593 (Δ = -0.056), median 0.553 vs 0.613.
  - PX: 0.460 vs 0.522 (Δ = -0.063), medians 0.461 vs 0.513.
  - PZ: 0.485 vs 0.526 (Δ = -0.041), medians 0.477 vs 0.531.
  - PAA: 0.438 vs 0.484 (Δ = -0.046), medians 0.411 vs 0.492.
- **Some populations show slightly higher purity in dense environments:**
  - PH: 0.458 vs 0.415 (Δ = +0.043), medians 0.458 vs 0.411.
  - PK: 0.449 vs 0.436 (Δ = +0.013), medians 0.423 vs 0.419.
- Statistical significance (after FDR):
  - PM, PX, PZ, PAA: FDR ≈ 5×10⁻⁴ or 0; rank-biserial ~0.18–0.28 (moderate).
  - PH: FDR = 0, rank-biserial ~-0.20, but note direction here is **dense more pure**.
  - PK: FDR ≈ 0.06, rank-biserial very small (~-0.037), so effect is marginal to null.

**Interpretation for the hypothesis:**
- There is **clear evidence that density is associated with systematic shifts in purity**, but:
  - The **direction differs by population**.
  - The **effect size is generally weaker** than for complexity.
- This heterogeneity is actually an asset: it will let you stratify populations into “density-associated purity increase” vs “density-associated purity decrease,” which is exactly what your next step calls for.

3. Evidence for “microenvironment-sensitive” subsets

If we define “microenvironment-sensitive” for this step as having:
- Robust complexity increase in dense vs sparse AND
- Robust purity change (FDR < 0.05) with a non-trivial effect size,

then:

- **PM, PX, PZ, PAA, PH** clearly qualify; they all have:
  - Strong complexity shifts (FDR=0, |rb|>0.26).
  - Significant purity shifts (FDR ≤ 0.0005), |rb| ~0.18–0.28 (moderate).
- **PK** is clearly density-sensitive for complexity, but only **weakly** so for purity (FDR ~0.06, very small effect). It’s borderline as a “purity-responding” population.

Within this subset:
- **Density-associated purity increase populations:** PH, PK (PH is strong, PK weak).
- **Density-associated purity decrease populations:** PM, PX, PZ, PAA.

This matches your plan to prioritize a subset of microenvironment-sensitive populations based on the **direction** of purity change.

4. How this informs the next steps of your plan

Step 2 (summary/classification table):
- You already have the core quantities:
  - Δ mean and median complexity and purity.
  - Rank-biserial effect sizes.
  - FDR-adjusted p-values.
- You can now **add a simple classification column**, e.g.:
  - `complexity_trend_in_dense`: “higher” for all 6 populations.
  - `purity_trend_in_dense`: 
    - “higher” for PH (and maybe PK if you include trends regardless of FDR).
    - “lower” for PM, PX, PZ, PAA.
  - `density_sensitivity_class` (example categories):
    - “Complex↑ + Purity↑” (PH; maybe PK borderline).
    - “Complex↑ + Purity↓” (PM, PX, PZ, PAA).
    - “Complex↑ only” (PK, if you require FDR < 0.05 for purity).
- This classification will be useful for prioritizing which populations to probe in the neighborhood-composition models and gene-level associations as representing distinct biological archetypes:
  - Complex↑ + Purity↓: possibly more heterogeneous or transitional states in dense niches.
  - Complex↑ + Purity↑: possibly more “refined” or lineage-pure states in dense niches.

Step 3 (neighborhood composition models):
- Use the `density_stratum_k15` now stored in `adata.obs` to:
  - Check that dense vs sparse cells are not strongly confounded with just one or two sections in each Sample_ID (you stratified within sample, which helps, but it’s worth verifying).
- For regression of complexity and purity on neighbor fractions:
  - Consider modeling **separately per population**, especially focusing on:
    - One “Complex↑ + Purity↑” population: **PH** as the primary exemplar.
    - One or more “Complex↑ + Purity↓” populations: **PM and PZ** are attractive because they show some of the strongest effects.
  - Include:
    - `local_density_k15`, `log10(UMI Count)`, and `Sample_ID` as you planned.
    - A small, interpretable set of neighbor fractions (e.g. the top few neighbor types by prevalence or by univariate association with complexity/purity).
  - A key question: **Does neighbor composition explain residual variance in complexity/purity beyond density itself?** You now know there’s a strong main effect of density; the models can clarify which neighbors augment or suppress that effect.

Step 4 (gene-level density associations):
- The strong marginal complexity differences across density strata mean there is likely a **rich set of genes** whose expression tracks density after controlling for UMIs and sample.
- To keep things distinct and interpretable:
  - Again, prioritize **PH vs PM/PZ/PAA** as contrasting examples when you look for:
    - Density-associated “maturation” programs (e.g., genes increasing with density in high-purity dense populations like PH).
    - Density-associated “stress/heterogeneity” programs (genes changing with density in purity-decreasing populations like PM/PZ/PAA).
- Once you regress out log10(UMI Count) and Sample_ID, you can:
  - Compute Spearman correlations between residualized expression and `local_density_k15`.
  - Count genes with FDR<0.1 or <0.05.
  - For each population, separately summarize:
    - #genes up with density vs down with density.
    - A few top genes by correlation magnitude to see whether they resemble maturation vs stress markers (within the constraints of your limited panel).

5. Methodological considerations / potential refinements

- **UMI Count is not yet controlled** in these dense–sparse comparisons. Because dense cells could systematically have higher UMIs:
  - The observed complexity differences might partly reflect coverage differences.
  - However, the effect sizes are so large that even after controlling for UMI in later regression, **you would likely retain strong density effects**.
  - In the follow-up regression models, explicitly check whether density remains a strong predictor of complexity after including log10(UMI Count); report that as a key validation of the hypothesis.
- **Sample-level structure:** you stratified dense/sparse within each Sample_ID, which is excellent; but:
  - Some samples for a given population were set to “mid” (if < 2×min_cells), and those samples don’t enter the test.
  - Check in downstream summaries whether your dense vs sparse cells are well-distributed across samples to rule out a single-sample driver.

6. Hypothesis status

Based on this step alone:

- **Supported:** For all six targeted populations, dense microenvironments are associated with meaningfully higher transcriptional complexity. This is fully in line with the hypothesis and quite robust statistically.
- **Partially but interestingly mixed:** For purity, the microenvironment effect is **population-specific**:
  - Some populations (PM, PX, PZ, PAA) clearly show **lower purity** in dense niches.
  - Others (PH, and weakly PK) show **higher purity** in dense niches.
- This heterogeneity is not a failure of the hypothesis; it is **exactly the kind of microenvironment-dependent, population-specific behavior** you want to uncover, and it creates a natural contrast set for subsequent neighborhood and gene-level analyses.

In summary, this step has:
- Clearly validated density–complexity coupling in the key populations.
- Exposed a nuanced, population-dependent pattern for purity that you can now formalize in your classification and then mechanistically probe via neighborhood composition and gene-level associations.

## Next Steps
Step 1: For key populations PM, PX, PZ, PAA, PH, and PK, quantify how selected neighborhood-composition features (neighbor_frac_*_k15) associate with transcriptional complexity and purity after controlling for local_density_k15, log10(UMI Count), and Sample_ID fixed effects, reporting per-population regression coefficients, p-values, and FDR-adjusted significance for each neighbor type.
Step 2: Summarize these models in a consolidated text table that, for each key population and outcome (complexity, purity), lists the neighbor types with robust positive or negative associations after correction, and classify populations into archetypes (e.g., density-driven complexity with purity loss vs density-driven complexity with purity gain) and highlight which neighbor types reinforce or oppose these trends.
Step 3: Within the same key populations, identify genes whose expression is significantly associated with local_density_k15 after regressing out log10(UMI Count) and Sample_ID (without including neighbor-composition terms), using per-gene Spearman correlations and FDR correction, and summarize per-population counts and top positive/negative density-associated genes to distinguish likely maturation-like vs stress/interaction-like programs, contrasting populations where dense niches increase vs decrease purity.

## This code fits, within key cardiac populations, multiple linear regression models relating transcriptional complexity and purity to local density, UMI depth, sample fixed effects, and prevalent neighborhood-composition fractions, using a numerically stable pseudo-inverse for standard errors and applying BH-FDR across all neighbor effects. It extracts per-neighbor coefficients, p-values, and FDR-adjusted significance for complexity and purity, storing a tidy summary table in adata.uns for downstream archetype summarization.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import numpy.linalg as LA

# We assume `adata` is already in memory and that prior steps have created:
#  - adata.obs['local_density_k15']
#  - adata.obs['UMI Count'], ['Complexity'], ['Purity'], ['Populations'], ['Sample_ID']
#  - neighbor_frac_*_k15 columns (fraction of each Populations label among k=15 spatial neighbors)
#  - dense/sparse labels are not required here, we model density as continuous.

# 1. Basic checks and setup
required_obs = ['Populations', 'Sample_ID', 'UMI Count', 'Complexity', 'Purity', 'local_density_k15']
missing = [c for c in required_obs if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Missing required columns in adata.obs: {missing}")

obs = adata.obs.copy()
obs['Populations'] = obs['Populations'].astype(str)
obs['Sample_ID'] = obs['Sample_ID'].astype(str)

for col in ['UMI Count', 'Complexity', 'Purity', 'local_density_k15']:
    obs[col] = obs[col].astype(float)

# Key populations: focus on those with strong density–complexity links and clear purity shifts
key_pops = ['PM', 'PX', 'PZ', 'PAA', 'PH', 'PK']
key_pops = [p for p in key_pops if p in obs['Populations'].unique()]
if not key_pops:
    raise ValueError("None of the expected key populations (PM, PX, PZ, PAA, PH, PK) are present in adata.obs['Populations'].")

print("Key populations included in neighborhood-composition modeling:", key_pops)

# Identify all neighbor-fraction columns
neighbor_cols = [c for c in obs.columns if c.startswith('neighbor_frac_') and c.endswith('_k15')]
if not neighbor_cols:
    raise ValueError("No neighbor_frac_*_k15 columns found in adata.obs; neighborhood composition was not computed.")

# 2. Helper for FDR correction

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = pvals.size
    if n == 0:
        return pvals
    order = np.argsort(pvals)
    ranked = pvals[order]
    qvals = np.empty(n, dtype=float)
    prev_q = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        q = ranked[i] * n / rank
        if q > prev_q:
            q = prev_q
        prev_q = q
        qvals[i] = q
    out = np.empty(n, dtype=float)
    out[order] = qvals
    return out

# 3. Multiple linear regression via ordinary least squares using numpy.linalg.lstsq
#    outcome ~ beta0 + beta_density * density + beta_logUMI * log10(UMI) + Sample_ID fixed effects + sum_j beta_j * neighbor_frac_j

results_rows = []

for pop in key_pops:
    sub = obs[obs['Populations'] == pop].copy()
    n_cells = sub.shape[0]
    if n_cells < 300:
        # Maintain a minimum for stability
        print(f"Skipping population {pop}: too few cells for stable regression (n={n_cells}).")
        continue

    print(f"\nFitting neighborhood-composition models for population {pop} (n={n_cells})")

    # Core covariates
    density = sub['local_density_k15'].values.astype(float)
    umi = sub['UMI Count'].values.astype(float)
    log_umi = np.log10(umi + 1.0)
    complexity = sub['Complexity'].values.astype(float)
    purity = sub['Purity'].values.astype(float)

    # Sample_ID fixed effects as one-hot (drop_first to avoid collinearity with intercept)
    sample_dummies = pd.get_dummies(sub['Sample_ID'], drop_first=True)

    # Select neighbor fractions that are actually present around this population
    # Criterion: mean neighbor fraction >= 0.05 in this population; this both reduces dimensionality
    # and mitigates collinearity among fractions that approximately sum to 1.
    sub_neighbor_means = sub[neighbor_cols].mean(axis=0)
    selected_neighbor_cols = sub_neighbor_means[sub_neighbor_means >= 0.05].index.tolist()

    if len(selected_neighbor_cols) == 0:
        print(f"  No neighbor types with mean fraction >= 0.05 around {pop}; skipping neighbor modeling.")
        continue

    # Construct design matrix: intercept + density + logUMI + sample FEs + selected neighbor fractions
    X_parts = []
    # intercept
    intercept = np.ones((n_cells, 1), dtype=float)
    X_parts.append(intercept)
    # density and logUMI
    X_parts.append(density.reshape(-1, 1))
    X_parts.append(log_umi.reshape(-1, 1))
    # sample dummies
    if sample_dummies.shape[1] > 0:
        X_parts.append(sample_dummies.values.astype(float))
    # neighbor fractions
    X_parts.append(sub[selected_neighbor_cols].values.astype(float))

    X = np.concatenate(X_parts, axis=1)

    # Keep track of column names in X aligned with betas
    design_colnames = (['intercept', 'local_density_k15', 'log10_UMI'] +
                       list(sample_dummies.columns) + selected_neighbor_cols)

    # Helper to fit OLS and get coefficients and standard errors
    def fit_ols(Xmat, y):
        mask = np.isfinite(Xmat).all(axis=1) & np.isfinite(y)
        Xf = Xmat[mask]
        yf = y[mask]
        n, p = Xf.shape
        if n <= p + 5:
            return None  # too few df
        beta, residuals, rank, s = LA.lstsq(Xf, yf, rcond=None)
        if rank < p:
            print(f"  Warning: design matrix rank ({rank}) < number of predictors ({p}); SEs may be unstable.")
        if residuals.size == 0:
            # Compute residual sum of squares manually if lstsq did not return it
            y_pred = Xf.dot(beta)
            rss = np.sum((yf - y_pred) ** 2)
        else:
            rss = residuals[0]
        dof = n - p
        sigma2 = rss / max(dof, 1)
        # covariance matrix of betas: sigma^2 * (X^T X)^{+}
        XtX = Xf.T.dot(Xf)
        XtX_inv = LA.pinv(XtX)
        se_beta = np.sqrt(np.diag(XtX_inv) * sigma2)
        t_stats = beta / se_beta
        p_vals = 2 * stats.t.sf(np.abs(t_stats), df=dof)
        return beta, se_beta, t_stats, p_vals, dof

    # Fit for complexity
    fit_c = fit_ols(X, complexity)
    if fit_c is None:
        print(f"  Could not fit complexity model for {pop} (insufficient degrees of freedom).")
        continue
    beta_c, se_c, t_c, p_c, dof_c = fit_c

    # Fit for purity
    fit_p = fit_ols(X, purity)
    if fit_p is None:
        print(f"  Could not fit purity model for {pop} (insufficient degrees of freedom).")
        continue
    beta_p, se_p, t_p, p_p, dof_p = fit_p

    # Extract neighbor-related effects only (skip intercept, density, logUMI, and sample dummies)
    base_terms = ['intercept', 'local_density_k15', 'log10_UMI'] + list(sample_dummies.columns)
    base_idx = set(design_colnames.index(t) for t in base_terms if t in design_colnames)

    for j, col in enumerate(design_colnames):
        if j in base_idx:
            continue
        if col not in selected_neighbor_cols:
            continue

        # Common info
        neighbor_pop = col[len('neighbor_frac_'):-len('_k15')] if col.startswith('neighbor_frac_') and col.endswith('_k15') else col
        mean_frac = float(sub[col].mean())

        # Complexity effects
        beta_c_j = float(beta_c[j])
        se_c_j = float(se_c[j])
        t_c_j = float(t_c[j])
        p_c_j = float(p_c[j])

        # Purity effects
        beta_p_j = float(beta_p[j])
        se_p_j = float(se_p[j])
        t_p_j = float(t_p[j])
        p_p_j = float(p_p[j])

        results_rows.append({
            'Population': pop,
            'neighbor_population': neighbor_pop,
            'design_column': col,
            'n_cells': int(n_cells),
            'mean_neighbor_fraction': mean_frac,
            'beta_complexity': beta_c_j,
            'se_complexity': se_c_j,
            't_complexity': t_c_j,
            'raw_p_complexity': p_c_j,
            'beta_purity': beta_p_j,
            'se_purity': se_p_j,
            't_purity': t_p_j,
            'raw_p_purity': p_p_j,
            'dof_complexity_model': int(dof_c),
            'dof_purity_model': int(dof_p)
        })

# 4. Compile results and apply FDR correction within each outcome (complexity, purity)
if len(results_rows) == 0:
    print("\nNo neighbor-composition models were successfully fit; results table is empty.")
    neighbor_effects_df = pd.DataFrame()
else:
    neighbor_effects_df = pd.DataFrame(results_rows)

    # FDR across all Population–neighbor pairs for each outcome separately
    neighbor_effects_df['fdr_p_complexity'] = bh_fdr(neighbor_effects_df['raw_p_complexity'].values)
    neighbor_effects_df['fdr_p_purity'] = bh_fdr(neighbor_effects_df['raw_p_purity'].values)

    # Add simple direction labels
    def direction(beta):
        if np.isnan(beta):
            return 'NA'
        return 'positive' if beta > 0 else 'negative' if beta < 0 else 'zero'

    neighbor_effects_df['direction_complexity'] = neighbor_effects_df['beta_complexity'].apply(direction)
    neighbor_effects_df['direction_purity'] = neighbor_effects_df['beta_purity'].apply(direction)

    # Sort for readability: by Population, then by fdr_p_complexity, then by |beta_complexity|
    neighbor_effects_df = neighbor_effects_df.sort_values(
        ['Population', 'fdr_p_complexity', 'raw_p_complexity', 'raw_p_purity'],
        ascending=[True, True, True, True]
    )

# Store results for downstream summarization in the next step
adata.uns['neighbor_composition_effects_complexity_purity'] = neighbor_effects_df

print("\nNeighbor-composition effects on complexity and purity (first 60 rows, rounded):")
if neighbor_effects_df.empty:
    print("No significant models; DataFrame is empty.")
else:
    print(neighbor_effects_df.round(4).head(60).to_string(index=False))


Key populations included in neighborhood-composition modeling: ['PM', 'PX', 'PZ', 'PAA', 'PH', 'PK']

Fitting neighborhood-composition models for population PM (n=7417)

Fitting neighborhood-composition models for population PX (n=1562)

Fitting neighborhood-composition models for population PZ (n=1286)

Fitting neighborhood-composition models for population PAA (n=1027)

Fitting neighborhood-composition models for population PH (n=10887)

Fitting neighborhood-composition models for population PK (n=8540)

Neighbor-composition effects on complexity and purity (first 60 rows, rounded):
Population neighbor_population         design_column  n_cells  mean_neighbor_fraction  beta_complexity  se_complexity  t_complexity  raw_p_complexity  beta_purity  se_purity  t_purity  raw_p_purity  dof_complexity_model  dof_purity_model  fdr_p_complexity  fdr_p_purity direction_complexity direction_purity
       PAA                  PK  neighbor_frac_PK_k15     1027                  0.2395           4.00

### Agent Interpretation

These regression results strongly support the first part of your hypothesis: neighborhood composition has substantial, independent explanatory power for both complexity and purity after controlling for density, UMI depth, and sample.

Key patterns and how they inform next steps:

1. **Neighbor fractions clearly matter beyond density/UMI/sample**

   - Every key population (PM, PX, PZ, PAA, PH, PK) with sufficient cells shows multiple neighbor types with highly significant effects (FDR ≪ 0.05) on both complexity and purity.
   - This directly validates that “immediate neighborhood composition explains additional variability” — the neighbor_beta estimates are non‑trivial, often large in magnitude, and consistently significant.

   For example:
   - **PM**: neighbors PB, PS, PI, PM strongly *decrease* complexity and *increase* purity; PF neighbors do the opposite (↑complexity, ↓purity).
   - **PH**: most neighbor types (PH, PQ, PL, PE, PC, PD) strongly modulate outcomes, often in opposite ways for complexity vs purity.
   - **PK**: PK neighbors → ↑complexity and ↑purity; PF and PY neighbors → ↑complexity but ↓purity.
   - **PZ**: PK, PD, PF neighbors → ↑complexity but ↓purity; PB neighbors → ↓complexity and ↑purity.
   - **PAA**: PK, PF, PAA neighbors → ↑complexity; PI, PM, PW neighbors → ↓complexity. For purity, several neighbor types have significant but directionally varied effects.

2. **Clear divergence in direction of neighbor effects on complexity vs purity**

   Your hypothesis emphasizes that, especially in populations where dense niches increase vs decrease purity, neighbor effects might reinforce or oppose density-driven purity trends. Even without explicitly bringing in the density–purity interaction yet, you already see divergent patterns:

   - **PM**:
     - PB/PS/PI/PM neighbors:  
       - Complexity: strongly *negative*  
       - Purity: strongly *positive*  
       → Local enrichment for these neighbors appears to “clean up” the population transcriptionally (higher purity) while *reducing* complexity.
     - PF neighbors:  
       - Complexity: positive  
       - Purity: negative  
       → Suggests a niche where mixed/more complex expression comes at the cost of purity.
   - **PX**:
     - PK neighbors: ↑complexity, ↓purity  
     - PX neighbors: ↓complexity, ↑purity  
     - PR neighbors: ↓complexity, ↑purity  
     → Self‑neighbors (PX) and certain neighbors (PR) push towards “simpler & purer,” whereas PK neighbors push “complex & impure.”
   - **PZ**:
     - PK / PD / PF neighbors: ↑complexity, ↓purity  
     - PB neighbors: ↓complexity, ↑purity  
     → A very similar architecture to PX.
   - **PK**:
     - PK self‑neighbors: ↑complexity, ↑purity  
       → A niche where being surrounded by one’s own type simultaneously increases both metrics.
     - PF and PY neighbors: ↑complexity, ↓purity  
       → Suggests specific heterotypic contacts drive complexity without maintaining purity.
   - **PH**:
     - PE, PQ, PL neighbors: ↓complexity, ↑purity  
     - PC, PD neighbors: ↓complexity, ↓purity (for PC both negative; PD: small negative complexity, strongly negative purity)  
     - PH self‑neighbors: ↓complexity with almost zero purity effect  
     → For PH, many neighbors reduce complexity, but only subsets increase purity, implying a more nuanced microenvironmental structure.

   Overall, there is *no single universal pattern*: the same neighbor type can have opposite effects on different target populations, and complexity/purity often move in opposite directions.

3. **Evidence that neighbor effects are not trivially redundant with density**

   Since you included `local_density_k15` and log10(UMI) along with sample fixed effects, the neighbor coefficients are conditional on density and depth. The strong and varied neighbor betas (and their directions) imply neighbor composition is not simply a proxy for “dense vs sparse.” For instance, within PZ, both PK and PB neighbors are present at comparable mean fractions (~0.14 vs ~0.06) but with opposite complexity/purity signatures, which would not be captured by density alone.

   This aligns well with your core mechanistic hypothesis that microenvironmental composition explains extra variance beyond density.

4. **Implications for your planned archetype classification**

   The current step gives you exactly the ingredients you need to define archetypes in the next step:

   - For each **focal population**:
     - Identify which **neighbor types**:
       - Are strongly associated with **↑complexity & ↑purity** (e.g., PK self‑neighbors for PK).
       - **↑complexity & ↓purity** (e.g., PK neighbors for PX; PK/PD/PF neighbors for PZ).
       - **↓complexity & ↑purity** (e.g., PB/PS/PI/PM neighbors for PM; PE/PQ/PL neighbors for PH; PX and PR neighbors for PX; PB neighbors for PZ).
       - **↓complexity & ↓purity** (e.g., PC neighbors for PH; some others with both negative).
   - Then overlay this with each population’s **global density–purity relationship** (from your earlier analysis) to see:
     - In populations where dense niches tend to **increase purity**, are the predominant dense neighbors those that also show **↓complexity & ↑purity** signatures?  
     - In populations where dense niches tend to **decrease purity**, do dense neighbors match **↑complexity & ↓purity** patterns?

   From the visible subset:
   - **PX and PZ** are clear candidates for a “dense heterotypic mixing → high complexity, low purity” archetype (e.g., PK/PD/PF neighbors).
   - **PM** and **PH** seem to have neighbors that drive opposing complexity/purity combinations; depending on your earlier bulk density–purity trend for these populations, you can classify them as:
     - “Density-driven purity gain with complexity suppression” niches (PB/PS/PI neighbors in PM; PE/PQ/PL in PH).
     - Versus heterotypic neighbors that instead erode purity.

5. **Suggestions for immediate follow-ups within Step 1/2**

   To make the next summarization step more interpretable and robust:

   - **Threshold for “robust” neighbors**:  
     Define something like `|beta| > x` and `FDR < 0.05` and perhaps `mean_neighbor_fraction ≥ 0.05` (already applied) as criteria for “key neighbor interactions.”
   - **Compact signed classification**:  
     For each (population, neighbor_pop) pair, define a 4‑level category:
     - C+P+, C+P−, C−P+, C−P− based on the signs of `beta_complexity` and `beta_purity` for significant neighbors.
   - **Self vs non‑self neighbors**:  
     Self‑neighbors (e.g., PK neighbors of PK) behave differently from heterotypic ones and should likely be reported separately; these may correspond to “core tissue” vs “border/interfacial” zones.

6. **Caveats / checks before moving on**

   - **Collinearity**: Fractions over neighbors sum to ~1; you partially address this by only using neighbors with mean ≥ 0.05, but there will still be some collinearity. The extremely high t‑statistics (e.g., >20–40) suggest large effects but also that the model is quite constrained by large n. It might be useful later to:
     - Check variance inflation factors (VIFs) or correlations among selected neighbor predictors for a couple of populations (especially PH, PM).
   - **Effect size vs scale**: Complexity and purity scales matter. Before biological interpretation, standardize complexity/purity within population or at least document their scale so that a beta of, say, 4 units in complexity is interpretable.
   - **Non‑linearities**: Right now, effects are linear; if you later see saturation patterns (e.g., effects only at very high neighbor fraction), you may consider spline terms or binning, but this is secondary to your current objective.

7. **How this informs Step 3 (density–gene associations)**

   The neighbor patterns suggest specific **microenvironmental niches** that increase complexity in low‑purity contexts vs high‑purity contexts. In Step 3 (density–gene correlations without neighbor terms), these results can guide a **stratified interpretation**:

   - For populations like **PX and PZ**, where high‑complexity neighbors tend to lower purity, genes positively associated with density may reflect **stress/interaction-like or mixing/interface programs**.
   - For populations like **PM, PH**, where some neighbors promote high purity with lower complexity, density-associated genes may be more “maturation-like,” especially in nuclei where self or pure neighbors dominate.
   - For **PK**, where self-density appears to increase both complexity and purity, density-associated genes might reflect a “coherent maturation or specification” program characteristic of a robust tissue compartment.

   When you compute per-gene density–complexity associations in Step 3, consider:
   - Reporting results **separately for populations with different neighbor archetypes**, to test if the gene sets underlying density‑driven complexity in “pure” vs “mixed/impure” niches are qualitatively distinct.

In summary, this step provides strong quantitative support that neighborhood composition has substantial, directionally rich effects on complexity and purity across key cardiac populations, beyond density and UMI depth. The observed diversity of direction combinations across populations and neighbor types is exactly the kind of structure you need to define the proposed archetypes and then interpret density-associated gene programs in Step 3.

## Next Steps
Step 1: Summarize the fitted neighborhood-composition regression models by extracting, for each key population (PM, PX, PZ, PAA, PH, PK) and each neighbor type, those neighbor_frac_*_k15 features that have robust associations (|beta| above a small threshold and FDR<0.05) with complexity and/or purity after controlling for local_density_k15, log10(UMI Count), and Sample_ID fixed effects, and classify each neighbor effect into one of four directional categories (C+P+, C+P−, C−P+, C−P−).
Step 2: Integrate these neighbor-effect summaries with existing dense–sparse density results to build a consolidated text table that, for each key population, reports whether dense niches globally increase or decrease purity, counts of robust neighbor types in each directional category, and a final archetype label (e.g., 'Complex↑+Purity↓ with heterotypic C+P− neighbors' vs 'Complex↑+Purity↑ with self C+P+ neighbors'), highlighting how specific neighbor compositions reinforce or oppose the population’s global density–purity trend.

## This code summarizes, for each key cardiac population, which neighbor types have robust regression-based effects on transcriptional complexity and purity after controlling for density, UMI depth, and sample, and classifies each into joint complexity–purity direction categories (C+P+, C+P−, C−P+, C−P−). It filters neighbor fractions by prevalence, effect size, and FDR, then compiles per-population counts and detailed text descriptions to support downstream archetype labeling.

In [ ]:
import numpy as np
import pandas as pd

# Retrieve previously computed neighbor-composition regression results
if 'neighbor_composition_effects_complexity_purity' not in adata.uns:
    raise KeyError("Expected 'neighbor_composition_effects_complexity_purity' in adata.uns; run the neighbor-composition regression step first.")

neighbor_df = adata.uns['neighbor_composition_effects_complexity_purity']
if not isinstance(neighbor_df, pd.DataFrame):
    neighbor_df = pd.DataFrame(neighbor_df)

required_cols = [
    'Population', 'neighbor_population', 'mean_neighbor_fraction',
    'beta_complexity', 'beta_purity', 'fdr_p_complexity', 'fdr_p_purity'
]
missing = [c for c in required_cols if c not in neighbor_df.columns]
if missing:
    raise KeyError(f"Missing expected columns in neighbor effects table: {missing}")

# Focus on the six key populations
key_pops = ['PM', 'PX', 'PZ', 'PAA', 'PH', 'PK']
key_pops = [p for p in key_pops if p in neighbor_df['Population'].unique()]
if not key_pops:
    raise ValueError("None of the key populations (PM, PX, PZ, PAA, PH, PK) are present in the neighbor effects table.")

# Define thresholds for 'robust' neighbor effects
fdr_thresh = 0.05
beta_min_mag = 0.05  # require at least a modest absolute effect size
mean_frac_min = 0.05  # keep neighbors that are reasonably prevalent

rows = []

# Helper to categorize sign combinations for complexity (C) and purity (P)
def combo_label(beta_c, beta_p):
    if np.isnan(beta_c) or np.isnan(beta_p):
        return 'NA'
    c_sign = '+' if beta_c > 0 else '-' if beta_c < 0 else '0'
    p_sign = '+' if beta_p > 0 else '-' if beta_p < 0 else '0'
    return f"C{c_sign}P{p_sign}"

for pop in key_pops:
    sub = neighbor_df[neighbor_df['Population'] == pop].copy()
    if sub.empty:
        continue

    # Filter to robust neighbor effects for at least one outcome
    mask_robust = (
        (sub['mean_neighbor_fraction'] >= mean_frac_min) &
        (
            ((sub['fdr_p_complexity'] < fdr_thresh) & (sub['beta_complexity'].abs() >= beta_min_mag)) |
            ((sub['fdr_p_purity'] < fdr_thresh) & (sub['beta_purity'].abs() >= beta_min_mag))
        )
    )
    sub_robust = sub[mask_robust].copy()

    if sub_robust.empty:
        rows.append({
            'Population': pop,
            'n_robust_neighbors': 0,
            'n_C+P+': 0,
            'n_C+P-': 0,
            'n_C-P+': 0,
            'n_C-P-': 0,
            'n_other_or_zero': 0,
            'robust_neighbors_detail': 'None'
        })
        continue

    # Assign directional categories based on signs of beta_complexity and beta_purity
    sub_robust = sub_robust.copy()
    sub_robust.loc[:, 'direction_combo'] = [combo_label(bc, bp) for bc, bp in zip(sub_robust['beta_complexity'], sub_robust['beta_purity'])]

    # Count each directional category
    counts = sub_robust['direction_combo'].value_counts().to_dict()
    n_cpp = counts.get('C+P+', 0)
    n_cpm = counts.get('C+P-', 0)
    n_cmp = counts.get('C-P+', 0)
    n_cmm = counts.get('C-P-', 0)

    # Count neighbors that are significant only in one outcome or have zero-ish betas
    n_other = int(sub_robust.shape[0] - (n_cpp + n_cpm + n_cmp + n_cmm))

    # Build a compact textual summary of robust neighbors for this population
    detail_entries = []
    for _, r in sub_robust.sort_values(['direction_combo', 'neighbor_population']).iterrows():
        detail_entries.append(
            f"{r['neighbor_population']}({r['direction_combo']}, mf={r['mean_neighbor_fraction']:.2f}, "
            f"bC={r['beta_complexity']:.2f}, bP={r['beta_purity']:.2f}, "
            f"FDRc={r['fdr_p_complexity']:.2e}, FDRp={r['fdr_p_purity']:.2e})"
        )

    rows.append({
        'Population': pop,
        'n_robust_neighbors': int(sub_robust.shape[0]),
        'n_C+P+': int(n_cpp),
        'n_C+P-': int(n_cpm),
        'n_C-P+': int(n_cmp),
        'n_C-P-': int(n_cmm),
        'n_other_or_zero': int(n_other),
        'robust_neighbors_detail': '; '.join(detail_entries)
    })

# Compile the per-population neighbor-summary table
neighbor_summary_df = pd.DataFrame(rows)
adata.uns['neighbor_composition_directional_summary'] = neighbor_summary_df

print("Directional summary of robust neighbor-composition effects on complexity and purity (per key population):")
if neighbor_summary_df.empty:
    print("No robust neighbor effects identified under the specified thresholds.")
else:
    print(neighbor_summary_df.to_string(index=False))

Directional summary of robust neighbor-composition effects on complexity and purity (per key population):
Population  n_robust_neighbors  n_C+P+  n_C+P-  n_C-P+  n_C-P-  n_other_or_zero                                                                                                                                                                                                                                                                                                                                                                                                   robust_neighbors_detail
        PM                   5       0       1       4       0                0                                                                    PF(C+P-, mf=0.07, bC=1.05, bP=-0.05, FDRc=4.71e-06, FDRp=4.79e-06); PB(C-P+, mf=0.26, bC=-5.66, bP=0.30, FDRc=0.00e+00, FDRp=0.00e+00); PI(C-P+, mf=0.17, bC=-3.46, bP=0.19, FDRc=2.65e-168, FDRp=6.20e-203); PM(C-P+, mf=0.21, bC=-3.79, bP=0.09, FDRc=1.87e-102

### Agent Interpretation

These neighbor-composition summaries strongly support the core hypothesis: for the strongly density–complexity–coupled cardiac populations, certain neighbor types explain additional variation in complexity and purity beyond density/UMI/sample, and the directions of these neighbor effects clearly differ between populations whose dense niches tend to purify vs decontaminate.

Key takeaways and how they relate to the hypothesis:

1. **Evidence that neighborhood composition matters beyond density/UMI/sample**

   - Every key population (PM, PX, PZ, PAA, PH, PK) has multiple “robust” neighbors whose fractions significantly affect complexity and/or purity under the regression model that already controls for local density, log10(UMIs), and Sample_ID.
   - The effect sizes are large in many cases (e.g., PM–PB bC = −5.66, PS bC = −7.64; PX–PK bC = 8.49; PZ–PD bC = 5.36), which is hard to explain as residual technical noise.
   - This directly validates the first part of the hypothesis: *immediate neighborhood composition explains additional variability* in complexity/purity over and above density and technical covariates.

2. **Distinct “archetypes” of neighborhood effects across key populations**

   Even just from the directional counts and details, the six populations are clearly not behaving uniformly. This is exactly what you want for the “archetype” step in your plan.

   I’d summarize each population’s neighbor-effect pattern as:

   **PM: Complexity↓, Purity↑ neighbors dominate (C−P+)**

   - 5 robust neighbors: 1 C+P− (PF), 4 C−P+ (PB, PI, PM self, PS), 0 C−P−.
   - Interpretation: PM is surrounded mainly by neighbor types that **decrease complexity but increase purity**. This is consistent with “cleaner but simpler” PM niches when surrounded by PB/PI/PS/PM.
   - PF is an interesting outlier: **C+P−**, so higher PF fraction makes PM **more complex but less pure**, a classic contamination-like signal.

   **PX: Mixed, including self C−P+ and heterotypic C+P− and C−P−**

   - 4 robust neighbors: 1 C+P− (PK), 2 C−P+ (PR, PX self), 1 C−P− (PY).
   - Self-neighbor PX(C−P+) suggests that **PX-rich microenvironments clean up purity at the cost of complexity** (or remove low-complexity contaminants in a way that net reduces internal complexity).
   - PK(C+P−) is the opposite: **PK neighbors make PX more complex and less pure**, again a contamination-like profile.
   - PY(C−P−) is a “toxic” neighbor: both complexity and purity go down when PY fraction is higher.

   **PZ: Strong split between C+P− and C−P+ neighbors + some C−P−**

   - 6 robust neighbors: 3 C+P− (PD, PF, PK), 1 C−P+ (PB), 2 C−P− (PM, PX).
   - PD/PF/PK: increase complexity and decrease purity for PZ; PB: decrease complexity but increase purity.
   - PM and PX neighbors jointly **suppress both complexity and purity** in PZ (C−P−).
   - This looks like a niche where “vascular-like” or otherwise distinct neighbors (PD/PF/PK) drive a more complex, contaminated PZ state, while PB-driven environments enforce a simpler, purer PZ.

   **PAA: The most balanced and bidirectional pattern**

   - 6 robust neighbors: 1 C+P+ (PK), 2 C+P− (self PAA, PF), 3 C−P+ (PI, PM, PW).
   - PK(C+P+) is notable: PAA in PK-rich neighborhoods becomes **both more complex and more pure** (bP is small but significant: 0.01, FDRp ~0.8 says purity not formally significant though, so interpret cautiously).
   - PAA self and PF: C+P−, suggesting **autologous and PF neighbors boost complexity at the cost of purity**.
   - PI/PM/PW neighbors: C−P+, shifting PAA toward **less complexity but higher purity**.
   - Overall PAA is a clean example where neighbor types pull it in opposite directions: some neighbors reinforce contamination-like profiles (C+P−), others enforce a “stripped-down but pure” state (C−P+).

   **PH: Almost entirely C−P+, with a subset of C−P−**

   - 6 robust neighbors: 4 C−P+ (PE, PH self, PL, PQ), 2 C−P− (PC, PD).
   - Dominant signature is **neighbors that lower complexity but increase purity** or at least not reduce it (PE, PL, PQ; PH self has bP≈0).
   - PC and PD are “bad” neighbors that suppress both complexity and purity.
   - This fits a niche where dense PH neighborhoods or PH with specific supportive neighbors tend to be **purified but transcriptionally streamlined**, consistent with density–purity increasing.

   **PK: Self-purifying with C+P+ self neighbors, but PF/PY are C+P−**

   - 3 robust neighbors: 1 C+P+ (PK self), 2 C+P− (PF, PY).
   - PK self-neighbors: both complexity and purity increase together; this is unusual compared with PM/PH, which see self often in C−P+ territory.
   - PF and PY neighbors for PK: standard contamination signature, C+P− (though PY’s purity effect is small and not significant at FDR<0.05).
   - This supports the idea that **dense PK microenvironments actually increase purity**, unlike populations where dense niches lead to contamination.

3. **Link to global density–purity trends (the second part of the hypothesis)**

   You already know which of these key populations had **density–purity coupling that was positive vs negative** from the earlier analysis (not shown here, but referenced in the hypothesis).

   The patterns above give you exactly what you need for the consolidated “archetype” table:

   - For populations where **denser microenvironments decreased purity** globally, you are seeing many **C+P− neighbors** (PD/PF/PK for PZ; PAA self and PF; PF and PK for PX; PF for PM; PF/PY for PK). These neighbor types are natural mechanistic candidates for the global density–purity decrease: they increase complexity and decrease purity in those populations.
   - For populations where **denser microenvironments increased purity**, the dominant neighbor categories are often **C−P+** and self C−P+ or C+P+ (PM self C−P+, PX self C−P+, PH self C−P+, PK self C+P+). These neighbors are consistent with density-driven purification.

   The important “directional difference” component of the hypothesis is supported:
   - PM & PH: dominated by **C−P+ self and heterotypic neighbors**; densification likely cleans them (purity↑) via these neighbors.
   - PZ & PAA: balanced or skewed toward **C+P− heterotypic neighbors** that increase complexity and hurt purity; dense PZ/PAA niches might be more contaminated or mixed.
   - PK: distinctive **C+P+ self effect**, meaning densification around same-type neighbors is both complex and pure, a qualitatively different regime.

4. **How to iterate and refine in the next step**

   For the planned “consolidated text table / archetype label” step, you’re already in good shape. Here are concrete refinements that will make that summary maximally interpretable and distinct from the paper:

   1) **Explicitly align neighbor categories with global density–purity slopes**

   - For each population, bring in:
     - Global dense vs sparse purity contrast (sign and effect size from previous analysis).
     - Global dense vs sparse complexity contrast.
   - Then classify each population into one of a few density archetypes:
     - “Dense → Purity↑, Complexity↓”,
     - “Dense → Purity↓, Complexity↑”,
     - “Dense → Purity↑ and Complexity↑”,
     - “Dense → Purity↓ and Complexity↓”.
   - Place your neighbor-direction counts underneath that, highlighting whether the dominant neighbor categories **reinforce** or **oppose** the global density trend. For instance:
     - If PM has dense→Purity↑Complexity↓ and its neighbors are mostly C−P+, that’s reinforcing.
     - If PK has dense→Purity↑Complexity↑ and self neighbors are C+P+, that’s perfectly aligned.

   2) **Distinguish self vs heterotypic neighbors**

   - In the summary, split robust neighbors into:
     - self (Population == neighbor_population),
     - other cardiac populations,
     - non-cardiac (?) if such label distinctions exist.
   - This will let you state things like:
     - “PM purification is mostly driven by self and PB/PI/PS neighbors (all C−P+), suggesting homotypic clustering plus a specific microenvironment drives PM into a purified but low-complexity state.”
     - “PK is unique in having **self C+P+** while PF/PY are C+P−, indicating that dense PK clusters are intrinsically high-quality, whereas certain heterotypic neighbors introduce contamination-like complexity.”

   3) **Weight by mean_neighbor_fraction to avoid overinterpreting rare neighbors**

   - You’re already filtering with mean_frac ≥ 0.05, which is sensible. For the archetype table:
     - Report both counts and **sum of mean_neighbor_fraction** per directional category (C+P+, C+P−, …).
     - This avoids giving equal weight to a rare PF(C+P−) vs a very prevalent PX(C−P+) in PX, for example.
   - You could also flag “dominant” neighbors as those with mean_neighbor_fraction ≥ some higher threshold (e.g., 0.15–0.20) and treat them as primary niches.

   4) **Summarize neighbor “roles” within each population**

   Instead of just counts, add a short interpretive label per population:

   - PM: “Complexity↓+Purity↑ neighbor regime dominated by PB/PI/PS/PM, with PF as a C+P− contaminant neighbor.”
   - PX: “Mixed regime; self PX and PR drive C−P+, but PK neighbors are C+P− and PY is C−P−, indicating both purifying and contaminating niches.”
   - PZ: “Strong C+P− neighbors (PD/PF/PK) aligned with increased complexity and contamination in dense PZ regions; PB enforces a C−P+ niche.”
   - PAA: “Balanced with both C+P− and C−P+ neighbors; PF and self PAA likely mediate complex, contaminated niches, while PI/PM/PW drive simpler, purer PAA states.”
   - PH: “Predominantly C−P+ neighbors (self, PE, PL, PQ), suggesting density-mediated purification with reduced complexity, punctuated by PC/PD niches that depress both.”
   - PK: “Self-driven C+P+ regime; PF/PY neighbors introduce standard C+P− contamination-like effects, making PK unique among key populations.”

   These textual archetypes would be the direct deliverable envisioned in your second planned step and are clearly distinct from the original paper, since they highlight abstract density–purity–complexity archetypes rather than named biological cell types or pathways.

5. **Optional further checks (if you want to strengthen the claim)**

   - **Residual plots / partial R²:** Confirm that adding neighbor fractions significantly increases explained variance in complexity and purity beyond density/UMI/sample, at least for representative populations (say, PM and PK). This would quantitatively back the statement that neighbor composition explains additional variability.
   - **Robustness across k:** If you computed neighbor composition at other k (e.g. k10, k20), test whether the sign of the key neighbor effects (e.g., PM–PB, PK–PK) is stable. You don’t need a full re-analysis; a quick comparison would strengthen the spatial interpretation.
   - **Visual sanity checks:** For a few key high-impact neighbor relationships (e.g., PK self C+P+, PM–PS C−P+), make 2D spatial maps coloring by:
     - focal population (e.g., PM) spatial distribution,
     - local neighbor_fraction of the relevant neighbor (PS),
     - fitted complexity/purity residuals.
     This will visually show the gradient your model is capturing and provide a sanity check against weird sample artifacts.

Overall, these model summaries do validate the core hypothesis: neighbor composition contributes nontrivially to complexity and purity after adjusting for density and technical factors, and the directionality of these neighbor effects differs across key cardiac populations in ways that you can now systematically encode in a density–purity–neighborhood “archetype” table.

## Next Steps
Step 1: Summarize the fitted neighborhood-composition regression models by extracting, for each key population (PM, PX, PZ, PAA, PH, PK) and each neighbor type, those neighbor_frac_*_k15 features that have robust associations (|beta| above a small threshold and FDR<0.05) with complexity and/or purity after controlling for local_density_k15, log10(UMI Count), and Sample_ID fixed effects, classify each neighbor effect into one of four directional categories (C+P+, C+P−, C−P+, C−P−), and tabulate per-population counts and text summaries that can be directly interpreted as neighborhood archetypes.

## This code takes the previously fitted neighborhood-composition regression results, filters for prevalent neighbor types with statistically robust effects on complexity and/or purity, assigns each such neighbor a joint sign category (C+P+, C+P−, C−P+, C−P−), and compiles per-population counts and text summaries so that neighborhood archetypes can be directly interpreted.

In [ ]:
import numpy as np
import pandas as pd

# Retrieve previously computed neighbor-composition regression results
if 'neighbor_composition_effects_complexity_purity' not in adata.uns:
    raise KeyError("Expected 'neighbor_composition_effects_complexity_purity' in adata.uns; run the neighbor-composition regression step first.")

neighbor_df = adata.uns['neighbor_composition_effects_complexity_purity']
if not isinstance(neighbor_df, pd.DataFrame):
    neighbor_df = pd.DataFrame(neighbor_df)

required_cols = [
    'Population', 'neighbor_population', 'mean_neighbor_fraction',
    'beta_complexity', 'beta_purity', 'fdr_p_complexity', 'fdr_p_purity'
]
missing = [c for c in required_cols if c not in neighbor_df.columns]
if missing:
    raise KeyError(f"Missing expected columns in neighbor effects table: {missing}")

# Focus on the six key populations
key_pops = ['PM', 'PX', 'PZ', 'PAA', 'PH', 'PK']
key_pops = [p for p in key_pops if p in neighbor_df['Population'].unique()]
if not key_pops:
    raise ValueError("None of the key populations (PM, PX, PZ, PAA, PH, PK) are present in the neighbor effects table.")

# Define thresholds for 'robust' neighbor effects
fdr_thresh = 0.05
beta_min_mag = 0.05  # require at least a modest absolute effect size
mean_frac_min = 0.05  # keep neighbors that are reasonably prevalent

rows = []

# Helper to categorize sign combinations for complexity (C) and purity (P)
def combo_label(beta_c, beta_p):
    if np.isnan(beta_c) or np.isnan(beta_p):
        return 'NA'
    c_sign = '+' if beta_c > 0 else '-' if beta_c < 0 else '0'
    p_sign = '+' if beta_p > 0 else '-' if beta_p < 0 else '0'
    return f"C{c_sign}P{p_sign}"

for pop in key_pops:
    sub = neighbor_df[neighbor_df['Population'] == pop].copy()
    if sub.empty:
        continue

    # Filter to robust neighbor effects for at least one outcome
    mask_robust = (
        (sub['mean_neighbor_fraction'] >= mean_frac_min) &
        (
            ((sub['fdr_p_complexity'] < fdr_thresh) & (sub['beta_complexity'].abs() >= beta_min_mag)) |
            ((sub['fdr_p_purity'] < fdr_thresh) & (sub['beta_purity'].abs() >= beta_min_mag))
        )
    )
    sub_robust = sub[mask_robust].copy()

    if sub_robust.empty:
        rows.append({
            'Population': pop,
            'n_robust_neighbors': 0,
            'n_C+P+': 0,
            'n_C+P-': 0,
            'n_C-P+': 0,
            'n_C-P-': 0,
            'n_other_or_zero': 0,
            'robust_neighbors_detail': 'None'
        })
        continue

    # Assign directional categories based on signs of beta_complexity and beta_purity
    sub_robust = sub_robust.copy()
    sub_robust.loc[:, 'direction_combo'] = [combo_label(bc, bp) for bc, bp in zip(sub_robust['beta_complexity'], sub_robust['beta_purity'])]

    # Count each directional category
    counts = sub_robust['direction_combo'].value_counts().to_dict()
    n_cpp = counts.get('C+P+', 0)
    n_cpm = counts.get('C+P-', 0)
    n_cmp = counts.get('C-P+', 0)
    n_cmm = counts.get('C-P-', 0)

    # Count neighbors that are significant only in one outcome or have zero-ish betas
    n_other = int(sub_robust.shape[0] - (n_cpp + n_cpm + n_cmp + n_cmm))

    # Build a compact textual summary of robust neighbors for this population
    detail_entries = []
    for _, r in sub_robust.sort_values(['direction_combo', 'neighbor_population']).iterrows():
        detail_entries.append(
            f"{r['neighbor_population']}({r['direction_combo']}, mf={r['mean_neighbor_fraction']:.2f}, "
            f"bC={r['beta_complexity']:.2f}, bP={r['beta_purity']:.2f}, "
            f"FDRc={r['fdr_p_complexity']:.2e}, FDRp={r['fdr_p_purity']:.2e})"
        )

    rows.append({
        'Population': pop,
        'n_robust_neighbors': int(sub_robust.shape[0]),
        'n_C+P+': int(n_cpp),
        'n_C+P-': int(n_cpm),
        'n_C-P+': int(n_cmp),
        'n_C-P-': int(n_cmm),
        'n_other_or_zero': int(n_other),
        'robust_neighbors_detail': '; '.join(detail_entries)
    })

# Compile the per-population neighbor-summary table
neighbor_summary_df = pd.DataFrame(rows)
adata.uns['neighbor_composition_directional_summary'] = neighbor_summary_df

print("Directional summary of robust neighbor-composition effects on complexity and purity (per key population):")
if neighbor_summary_df.empty:
    print("No robust neighbor effects identified under the specified thresholds.")
else:
    print(neighbor_summary_df.to_string(index=False))

Directional summary of robust neighbor-composition effects on complexity and purity (per key population):
Population  n_robust_neighbors  n_C+P+  n_C+P-  n_C-P+  n_C-P-  n_other_or_zero                                                                                                                                                                                                                                                                                                                                                                                                   robust_neighbors_detail
        PM                   5       0       1       4       0                0                                                                    PF(C+P-, mf=0.07, bC=1.05, bP=-0.05, FDRc=4.71e-06, FDRp=4.79e-06); PB(C-P+, mf=0.26, bC=-5.66, bP=0.30, FDRc=0.00e+00, FDRp=0.00e+00); PI(C-P+, mf=0.17, bC=-3.46, bP=0.19, FDRc=2.65e-168, FDRp=6.20e-203); PM(C-P+, mf=0.21, bC=-3.79, bP=0.09, FDRc=1.87e-102

### Agent Interpretation

These results strongly support the core of your hypothesis: for the six focal cardiac populations, immediate neighborhood composition has robust, directionally interpretable effects on both complexity and purity, distinct from local density/UMI/sample effects. A few points and suggestions by population and for next steps:

1. Overall pattern relative to the hypothesis  
   - You detect many neighbor types with |beta| well above the 0.05 threshold and very strong FDR support in most key populations.  
   - All four directional classes (C+P+, C+P−, C−P+, C−P−) appear across the six populations (though not equally), indicating that neighbor composition does modify complexity and purity in combinatorially rich ways.  
   - The presence of both “beneficial” neighbors (e.g., C−P+ for high-purity effect) and “detrimental” neighbors (e.g., C+P− or C−P−) suggests that microenvironments can either sharpen or erode transcriptional purity independent of density and UMI depth.

   What’s not yet explicit here is the link to *global* density–purity trends per population (i.e., which focal populations live in high‑density/high‑purity vs high‑density/low‑purity microenvironments). You seem to have that from earlier modeling; the next step is to overlay these neighbor-direction profiles on those population‑level “global sign” summaries.

2. Population-wise interpretation and promising directions  

   **PM**  
   - Robust neighbors: 5; all with reasonably common mean fractions (0.07–0.26).  
   - Neighbor archetype: overwhelmingly C−P+ (PB, PI, PM, PS) plus one C+P− (PF).  
   - Interpretation: PM’s microenvironment tends to follow a “complexity down, purity up” pattern when surrounded by itself (PM) and several other neighbors (PB, PI, PS). This is reminiscent of maturation/specialization: more constrained transcriptional programs but more cell-type purity. PF is an exception: it *increases* complexity while slightly *decreasing* purity.  
   - Follow‑up:  
     - Stratify PM cells by the relative abundance of PB/PI/PS vs PF neighbors and check whether this predicts transitions in marker expression, within-PM heterogeneity, or spatial niches.  
     - Compare these PM neighbor effects to PM’s global density–purity coupling: is PM a population where denser microenvironments raise or lower purity overall? If PM globally increases purity with density, the dominance of C−P+ neighbors is consistent with that; if not, it suggests micro‑heterogeneity.

   **PX**  
   - Robust neighbors: 4, including strong self‑neighbor effect (PX) and notable PK and PR.  
   - Archetype: mix of C−P+ (PR, PX), C+P− (PK), and one C−P− (PY).  
   - Self‑neighbor: PX(C−P+) suggests that PX cells in PX‑dense neighborhoods have reduced complexity but higher purity.  
   - PK(C+P−) is striking: a strongly positive complexity effect and strongly negative purity effect.  
   - Follow‑up:  
     - Ask whether regions enriched for PK neighbors around PX cells correspond to transitional or “immature/ambiguous” zones (highly complex, low purity).  
     - Map PX cells colored by PK and PR neighbor fractions; see if these correspond to different anatomical locales or border regions.  
     - Check whether global PX density–purity sign matches the dominant local pattern (C−P+ vs C+P− neighbors).

   **PZ**  
   - Robust neighbors: 6 with a balanced mix: 3 C+P− (PD, PF, PK), 1 C−P+ (PB), and 2 C−P− (PM, PX).  
   - Interpretation:  
     - PD/PF/PK microenvironments: complexity and impurity both increase (C+P−), potentially corresponding to mixed or transitional neighborhoods that “activate” PZ while eroding its purity.  
     - PB microenvironments: the reverse (C−P+), consistent with a more specialized, “pure” niche.  
     - PM/PX neighbors are broadly deleterious to both metrics (C−P−).  
   - Follow‑up:  
     - Use these as archetypal microenvironments: “activation/mixed” (PD/PF/PK), “refined/pure” (PB), and “suppressed” (PM/PX).  
     - Cluster PZ cells by their neighbor-fraction vectors and see if these clusters correspond to distinct spatial domains or complexity–purity regimes, then link back to earlier findings on PZ’s density–complexity coupling.

   **PAA**  
   - Robust neighbors: 6; this is the only focal population with a strong presence of C+P+ neighbors (PK, and to a lesser extent self PAA is C+P−).  
   - Archetype: one clear C+P+ (PK), two C+P− (PAA, PF), three C−P+ (PI, PM, PW).  
   - Interpretation:  
     - PK neighbors: slightly increase purity and strongly increase complexity (C+P+). This is a “beneficial high‑information, high‑purity” niche for PAA, unusual compared to other focal populations.  
     - PAA self‑neighbors: C+P− suggests that PAA-enriched patches boost PAA complexity but reduce purity, possibly indicating heterogeneity within PAA or interaction with unmodeled subtleties.  
     - PI/PM/PW: largely C−P+; these neighbors push PAA toward more specialized, cleaner states.  
   - Follow‑up:  
     - Contrast PAA cells in PK-rich vs PI/PM/PW-rich microenvironments; ask whether they show gene-expression signatures resembling different maturation stages.  
     - Align these neighbor archetypes with PAA’s global density–purity trend: if PAA is globally in high-density/low-purity regimes, PK neighbors may mark a distinct niche that breaks that trend.

   **PH**  
   - Robust neighbors: 6; no C+ effects, only C− categories (C−P+, C−P−).  
   - Archetype: 4 C−P+ (PE, PH, PL, PQ) and 2 C−P− (PC, PD).  
   - Interpretation:  
     - PH seems to sit in neighborhoods where complexity is consistently *depressed*, with purity either increased (PE/PH/PL/PQ) or decreased (PC/PD).  
     - Self‑neighbor PH(C−P+) indicates that PH‑dense regions are more “pure but transcriptionally minimal” niches.  
   - Follow‑up:  
     - Compare PH to earlier analyses of PH’s density–complexity coupling; this pattern is exactly the kind of “densely packed, low complexity but high purity” niche that your hypothesis predicts for maturing cell types.  
     - Explore whether the C−P− neighbors (PC, PD) mark boundaries or mixing zones that correlate with distinct spatial compartments.

   **PK**  
   - Robust neighbors: 3, all with positive effects on complexity (two C+P− and one C+P+).  
   - Archetype: heterotypic neighbors PF and PY are both C+P−; self PK is C+P+.  
   - Interpretation:  
     - PK’s own neighborhoods produce global increases in both complexity and purity (C+P+), suggesting PK-centric domains are “strong, defined states.”  
     - PF and PY neighbors add complexity but slightly reduce purity, a pattern similar to what you see for PK as a neighbor of PX and PZ, where PK tends to be C+P− for others. So PK seems to play a “complexity-raising, purity-blurring” role in multiple contexts.  
   - Follow‑up:  
     - Map PK’s spatial distribution and overlay its neighbor archetypes to see whether PK is a core of certain domains, with PF/PY marking more mixed peripheries.

3. How this bears on the main hypothesis  

   - The hypothesis centers on:  
     1) neighborhood composition explaining *additional* variation in complexity and purity beyond density/UMI/sample; and  
     2) the *sign* of neighbor effects differing between populations whose dense microenvironments globally raise vs lower purity.
   - Point (1) looks strongly supported: you have many robust neighbor effects across all six focal populations, with sizeable betas even after controlling for local_density_k15, log10(UMI), and Sample_ID.  
   - Point (2) is not yet explicitly tested in this step, but your table gives a clear way forward:  
     - For each focal population, you can compute the distribution of neighbor categories (e.g., fraction of robust neighbors that are C−P+, C+P−, etc.) and compare it between populations that, in prior modeling, showed density‑associated purity gains vs losses.  
     - If, for instance, global “purity‑raising with density” populations (e.g., PM, PH, PK if that’s what you found) are enriched for C−P+ and C+P+ neighbors, while “purity‑lowering” ones (e.g., PX, PZ, PAA, depending on prior results) are enriched for C+P− or C−P− neighbors, that would strongly validate the second part of the hypothesis. Your current summary hints this might be the case (PH and PM heavily C−P+; PZ and PX have substantial C+P− content), but you need the population‑level density–purity signs to make that explicit.

4. Suggested immediate next analytical steps  

   1. **Quantify neighbor‑class enrichment vs global density–purity sign**  
      - Retrieve, for each focal population, the sign and magnitude of the *global* density–purity slope from your earlier density–complexity–purity regressions.  
      - For each focal population, compute:  
        - Fraction of robust neighbors in each class (C+P+, C+P−, C−P+, C−P−).  
      - Test whether these fractions differ systematically between “density‑purity raising” vs “density‑purity lowering” populations (e.g., simple Fisher’s exact test or logistic model with population-level covariates).

   2. **Archetypal microenvironments as spatial niches**  
      - For each focal population, define archetypes based on the most common/significant neighbor types:  
        - e.g., for PZ: “PB‑rich” (C−P+), “PD/PF/PK‑rich” (C+P−), “PM/PX‑rich” (C−P−).  
      - Assign each focal cell to an archetype (e.g., by highest neighbor fraction among robust neighbors or by clustering the neighbor-fraction data) and map these archetypes onto spatial coordinates.  
      - Examine whether archetypes coincide with anatomical subregions, density bands, or known complexity–purity regimes.

   3. **Relate neighbor archetypes to gene expression programs**  
      - For each focal population, within each archetype, perform differential expression or at least signature scoring to see whether particular gene modules (e.g., developmental TFs or structural genes from the panel) associate with specific neighbor contexts, beyond what complexity alone would predict.  
      - This would connect the statistical neighbor effects to interpretable “programs” without overlapping the paper’s reported communities.

   4. **Check for symmetry and reciprocity of neighbor effects**  
      - Some pairs show interesting asymmetries:  
        - PK as neighbor is generally C+P− for others (PX, PZ, PAA) but PK self-neighbor is C+P+.  
        - PM is C−P+ for PM and PAA but C−P− for PZ.  
      - Systematically characterize such asymmetries; they provide nuanced evidence that neighbor effects are context‑dependent and not simply symmetric pairwise interactions.

   5. **Sensitivity and robustness checks (brief)**  
      - Repeat the summarization with slightly different thresholds (e.g., beta_min_mag = 0.1, mean_frac_min = 0.03) to ensure the main directional patterns (like PM/PH dominated by C−P+ self-neighbors, PZ with strong C+P− heterotypic neighbors, etc.) remain qualitatively stable.  
      - Confirm that including/excluding self‑neighbors does not invert your main conclusions.

In summary, the step you ran clearly identifies distinct, robust neighbor effects on complexity and purity for all six focal populations, giving you a solid foundation to define microenvironment “archetypes.” The next crucial piece is to explicitly tie these neighbor-direction profiles back to each population’s global density–purity coupling and to map archetypes into spatial and transcriptional space. That will let you directly confirm whether populations whose dense microenvironments globally raise purity are indeed enriched for “purity-raising” (especially C−P+ or C+P+) neighbor contexts, as hypothesized, and whether the opposite holds for purity-lowering populations.